# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d08_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # pile train
    shutil.copy(glob.glob("/kaggle/input/**/d08_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # pile test
    print("Kaggle: pile data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: pile data copied to ./drive/MyDrive/Regression/pile_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from scipy.stats import genpareto
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import warnings
import io
import time
import numpy as np
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795317515.343505      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795317536.408300      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (pile_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['F1', 'F2', 'Zv1', 'Zv2', 'phi', 'Th', 'L']

In [6]:
train_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (81, 8)
First 5 rows of training data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.80  1.21  4.90 -1.87  0.5  0.125  32   5.55
1  4.34  1.09  4.36 -1.95  0.5  0.125  45   6.07
2  4.71  0.81  5.10 -1.49  0.5  0.125  42   6.06
3  5.55  1.04  5.30 -1.46  0.5  0.125  38   5.94
4  3.08  0.95  2.82 -1.60  0.4  0.097  36   3.88

Shape of test data: (24, 8)
First 5 rows of test data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.69  1.06  4.70 -1.82  0.5  0.125  42   6.12
1  3.52  0.64  3.55 -1.03  0.4  0.097  45   4.26
2  3.70  0.49  3.43 -1.13  0.5  0.100  44   4.78
3  4.17  1.45  4.44 -0.21  0.5  0.125  16   5.78
4  4.33  2.04  4.75 -1.66  0.5  0.125  32   5.84


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (81, 7)
Shape of y_train: (81,)
Shape of X_test: (24, 7)
Shape of y_test: (24,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.006145   -0.12114279  0.17814132 -0.04043083 -0.57071458  0.40496686
  -0.73776147]
 [-0.37545934 -0.27898568 -0.27593219 -0.10402107 -0.57071458  0.40496686
   0.23089542]
 [-0.06851672 -0.64728574  0.3463167   0.2616228  -0.57071458  0.40496686
   0.00735922]
 [ 0.62832598 -0.34475355  0.51449207  0.28546914 -0.57071458  0.40496686
  -0.29068906]
 [-1.42072338 -0.46313571 -1.57088258  0.17418622 -2.53785844 -2.27277321
  -0.4397132 ]]

First five rows of normalized X_test:
[[-8.51082130e-02 -3.18446397e-01  9.96594816e-03 -6.86931582e-04
  -5.70714577e-01  4.04966863e-01  7.35921670e-03]
 [-1.05571054e+00 -8.70896496e-01 -9.57042459e-01  6.27266667e-01
  -2.53785844e+00 -2.27277321e+00  2.30895424e-01]
 [-9.06387106e-01 -1.06820010e+00 -1.05794768e+00  5.47778870e-01
  -5.70714577e-01 -1.98587249e+00  1.56383355e-01]
 [-5.16487026e-01  1.94542980e-01 -2.08662040e-01  1.27906661e+00
  -5.70714577e-01  4.04966863e-01 -1.92995458e+00]
 [-3.83

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=10):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test.to_numpy()[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test.to_numpy()[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.to_numpy().ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=10, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=10, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax


# FIX N2: the interpolating version below is NOT the NexCP weighted quantile (no +inf mass for the
# test point, linear interpolation between residuals). Renamed so it no longer overrides the
# corrected `weighted_quantile` in the helper cell; kept only for reference.
def _old_weighted_quantile_interp(values, weights, q):
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]
    cum_weights = np.cumsum(weights)
    cum_weights = cum_weights / cum_weights[-1]
    return np.interp(q, cum_weights, values)

In [13]:
# FIX N1-N9: helpers for the corrected NEXCP / online CP / COP / Extreme CP methods.
# Sources (see REFERENCES.md):
#   NexCP  - Barber, Candes, Ramdas & Tibshirani (2023), "Conformal prediction beyond exchangeability",
#            eq. (10)-(11) weighted split, eq. (12) weighted full, eq. (33) weighted jackknife+ / CV+.
#   J+aB   - Kim, Xu & Barber (2020), "Predictive inference is free with the jackknife+-after-bootstrap".
#   SF-OGD, SAOCP - Bhatnagar et al. (2023), ported line by line from the authors' code
#            github.com/salesforce/online_conformal (ogd.py, saocp.py).
#   FACI   - Gibbs & Candes (2024), ported from the same repo (faci.py); ACI - Gibbs & Candes (2021).
#   Extreme CP - Pasche, Lam & Engelke (2025), official R package ExtremeConformal (GPD_quantiles,
#            conformalize_GPD_max).
import math
from scipy.special import logsumexp

ALPHA = 0.1   # FIX M4: one miscoverage level for every method in this notebook (target coverage 90%)


# ---------------------------------------------------------------- N1: out-of-fold residuals
def _oof_fold_predictions(model_class, params, X_train, y_train, X_test, n_folds=5, seed=123):
    """Out-of-fold absolute residuals for every training row (each row scored by a model that never
    saw it), plus the fold models' predictions on X_test.
    Returns R (n,), mu_rows (n, n_test) = prediction at each test x of the fold model that left row i
    out, and fold_preds (K, n_test)."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    R = np.zeros(len(y))
    mu_rows = np.zeros((len(y), len(Xt)))
    fold_preds = []
    for tr, cal in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X):
        m = model_class(**params)
        m.fit(X[tr], y[tr])
        R[cal] = np.abs(y[cal] - np.asarray(m.predict(X[cal])).ravel())
        p = np.asarray(m.predict(Xt)).ravel()
        mu_rows[cal] = p[None, :]
        fold_preds.append(p)
    return R, mu_rows, np.asarray(fold_preds)


def _jab_oob(model_class, params, X_train, y_train, X_test, n_bootstrap, seed=123):
    """Jackknife+-after-bootstrap: each training row is scored only by the bootstrap models whose
    sample did NOT contain it (out-of-bag). Rows that were in every bootstrap sample are dropped."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    n = len(y)
    rng = np.random.default_rng(seed)
    in_bag = np.zeros((n_bootstrap, n), dtype=bool)
    P_tr = np.zeros((n_bootstrap, n))
    P_te = np.zeros((n_bootstrap, len(Xt)))
    for b in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        in_bag[b, idx] = True
        m = model_class(**params)
        m.fit(X[idx], y[idx])
        P_tr[b] = np.asarray(m.predict(X)).ravel()
        P_te[b] = np.asarray(m.predict(Xt)).ravel()
    oob = ~in_bag
    has = oob.any(axis=0)
    cnt = np.maximum(oob.sum(axis=0), 1)
    mu_tr = (P_tr * oob).sum(axis=0) / cnt
    R = np.abs(y - mu_tr)
    mu_rows = (oob.T.astype(float) @ P_te) / cnt[:, None]
    return R[has], mu_rows[has], P_te.mean(axis=0), has


# ---------------------------------------------------------------- N2: NexCP weighted quantile
def weighted_quantile(values, weights, q):
    """NexCP eq. (10)-(11): Q_q( sum_i w~_i delta_{R_i} + w~_{n+1} delta_{+inf} ),
    w~_i = w_i / (w_1+...+w_n+1). Returns the smallest value whose cumulative normalised weight reaches q
    (a step quantile, no interpolation), or +inf when the test point's own mass is needed.
    With all weights 1 this is the usual split-conformal ceil((n+1)(1-alpha))-th smallest residual.
    Was: normalised by w_1+...+w_n only (no test-point mass) and interpolated with np.interp."""
    v = np.asarray(values, dtype=float).ravel()
    w = np.asarray(weights, dtype=float).ravel()
    order = np.argsort(v)
    v, w = v[order], w[order]
    cum = np.cumsum(w) / (w.sum() + 1.0)
    k = np.searchsorted(cum, q - 1e-12, side="left")
    return np.inf if k >= len(v) else float(v[k])


CAP_NOTE = "interval capped: weighted quantile reached +inf (set to the largest calibration error)"


def _cap_q(q, R):
    if np.isfinite(q):
        return q, ""
    return float(np.max(R)), CAP_NOTE


def _jackknife_plus(mu_rows, R, w, alpha):
    """Weighted jackknife+ / CV+ (NexCP eq. 33): for each test point,
    lower = Q_alpha( sum w~_i delta_{mu_-i(x) - R_i} + w~_{n+1} delta_{-inf} ),
    upper = Q_{1-alpha}( sum w~_i delta_{mu_-i(x) + R_i} + w~_{n+1} delta_{+inf} ).
    Infinite bounds are capped at mu +/- max(R) and counted."""
    lo, hi, n_cap = [], [], 0
    for j in range(mu_rows.shape[1]):
        mu = mu_rows[:, j]
        l = -weighted_quantile(-(mu - R), w, 1 - alpha)
        h = weighted_quantile(mu + R, w, 1 - alpha)
        if not (np.isfinite(l) and np.isfinite(h)):
            n_cap += 1
            l = l if np.isfinite(l) else float(np.mean(mu) - np.max(R))
            h = h if np.isfinite(h) else float(np.mean(mu) + np.max(R))
        lo.append(l)
        hi.append(h)
    return np.asarray(lo), np.asarray(hi), n_cap


def _cap_count_note(n_cap, n):
    return "" if n_cap == 0 else f"{n_cap} of {n} intervals capped: weighted quantile reached +inf"


# ---------------------------------------------------------------- N3: NexCP full conformal (eq. 12)
def _nexcp_full(model_class, params, X_train, y_train, X_test, yhat_test, w, alpha, half_range,
                grid_size=50, label=""):
    """For each test x and each candidate y on a grid of `grid_size` values (prediction +/- half_range):
    refit on train + (x, y), compute all residuals R^y, and keep y if the test residual is not above
    the weighted (1-alpha) quantile of { R^y_i (weight w~_i), R^y_{n+1} (weight w~_{n+1}) }.
    The interval is [smallest kept y, largest kept y]."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    wt = np.asarray(w, dtype=float) / (np.sum(w) + 1.0)
    lo, hi, n_edge, n_empty = [], [], 0, 0
    t0 = time.time()
    for j in range(len(Xt)):
        grid = np.linspace(yhat_test[j] - half_range, yhat_test[j] + half_range, grid_size)
        X_aug = np.vstack([X, Xt[j:j + 1]])
        keep = []
        for yc in grid:
            y_aug = np.append(y, yc)
            m = model_class(**params)
            m.fit(X_aug, y_aug)
            r = np.abs(y_aug - np.asarray(m.predict(X_aug)).ravel())
            if wt[r[:-1] < r[-1]].sum() < 1 - alpha:
                keep.append(yc)
        if keep:
            lo.append(min(keep))
            hi.append(max(keep))
            n_edge += int(np.isclose(keep[0], grid[0]) or np.isclose(keep[-1], grid[-1]))
        else:
            lo.append(yhat_test[j])
            hi.append(yhat_test[j])
            n_empty += 1
        if (j + 1) % 5 == 0 or j == len(Xt) - 1:
            print(f"  NexCP-Full {label}: {j + 1}/{len(Xt)} test points, {(time.time() - t0) / 60:.1f} min")
    notes = []
    if n_edge:
        notes.append(f"{n_edge} of {len(Xt)} intervals reach the edge of the y-grid (true interval may be wider)")
    if n_empty:
        notes.append(f"{n_empty} empty sets (set to the point prediction)")
    return np.asarray(lo), np.asarray(hi), "; ".join(notes)


# ---------------------------------------------------------------- N4/N6: SF-OGD and SAOCP (salesforce/online_conformal)
def _pinball(y, yhat, q):
    return np.maximum(q * (y - yhat), (1 - q) * (yhat - y))


def _pinball_grad(y, yhat, q):
    return -q * (y > yhat) + (1 - q) * (y < yhat)


def _sf_ogd(calib, stream, alpha):
    """Scale-free OGD on the pinball loss (ogd.py ScaleFreeOGD): warm-started on the calibration
    residuals, then for each test point the margin is OUTPUT first and updated after its error is seen."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    state = {"delta": 0.0, "gn": 0.0}

    def update(s):
        g = _pinball_grad(abs(s), state["delta"], coverage)
        state["gn"] += g ** 2
        if state["gn"] != 0:
            state["delta"] = max(0.0, state["delta"] - scale / np.sqrt(3 * state["gn"]) * g)

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(state["delta"])
        update(s)
    return np.asarray(out)


class _SAOCPExpert:
    """saocp.py _OGD: scale-free OGD expert started at time t with lifetime g * 2^u."""

    def __init__(self, t, scale, alpha, yhat_0, g=8):
        self.scale = scale
        self.base_lr = scale / np.sqrt(3)
        self.alpha = alpha
        self.yhat = yhat_0
        self.grad_norm = 0
        u = 0
        while t % 2 == 0:
            t /= 2
            u += 1
        self.lifetime = g * 2 ** u
        self.z = 0
        self.wz = 0
        self.s_t = 0

    @property
    def expired(self):
        return self.s_t > self.lifetime

    def loss(self, y):
        return _pinball(y, self.yhat, 1 - self.alpha)

    @property
    def w(self):
        return 0 if self.s_t == 0 else self.z / self.s_t * (1 + self.wz)

    def update(self, y, meta_loss):
        w = self.w
        g = np.clip((meta_loss - self.loss(y)) / self.scale / max(self.alpha, 1 - self.alpha), -1 * (w > 0), 1)
        self.z += g
        self.wz += g * w
        self.s_t += 1
        grad = _pinball_grad(y, self.yhat, 1 - self.alpha)
        self.grad_norm += grad ** 2
        if self.grad_norm != 0:
            self.yhat = max(0, self.yhat - self.base_lr / np.sqrt(self.grad_norm) * grad)


def _saocp(calib, stream, alpha, lifetime=8):
    """saocp.py SAOCP (coin-betting combination of SF-OGD experts started at every step)."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    experts = {}
    state = {"t": 1}

    def get_p():
        if not experts:
            return {}
        prior = {t: 1 / (t ** 2 * (1 + np.floor(np.log2(t)))) for t in experts}
        z = sum(prior.values())
        prior = {t: v / z for t, v in prior.items()}
        p = {t: prior[t] * max(0, e.w) for t, e in experts.items()}
        sp = sum(p.values())
        return {t: v / sp for t, v in p.items()} if sp > 0 else prior

    def predict():
        p = get_p()
        return sum(p[t] * e.yhat for t, e in experts.items())

    def update(s):
        s = abs(s)
        s_hat = predict()
        for t in [k for k, e in experts.items() if e.expired]:
            experts.pop(t)
        experts[state["t"]] = _SAOCPExpert(state["t"], scale, 1 - coverage, yhat_0=s_hat, g=lifetime)
        meta_loss = _pinball(s, predict(), coverage)
        for e in experts.values():
            e.update(s, meta_loss)
        state["t"] += 1

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(predict())
        update(s)
    return np.asarray(out)


# ---------------------------------------------------------------- N5: FACI (faci.py) and ACI
def _faci(calib, stream, alpha):
    """Fully Adaptive Conformal Inference: 8 ACI experts with step sizes 0.001 * 2^k, combined with
    exponential weights on the pinball loss (Gibbs & Candes 2024, Algorithm 1; defaults of faci.py)."""
    coverage = 1 - alpha
    gammas = np.asarray([0.001 * 2 ** k for k in range(8)])
    k = len(gammas)
    I = 100
    sigma = 1 / (2 * I)
    a = 1 - coverage
    denom = ((1 - a) ** 2 * a ** 3 + a ** 2 * (1 - a) ** 3) / 3
    eta = np.sqrt(3 / I) * np.sqrt((np.log(I * k) + 2) / denom)
    alphas = np.full(k, a)
    log_w = np.zeros(k)
    residuals = list(np.abs(calib))
    out = []
    for s in np.abs(stream):
        w = np.exp(log_w - logsumexp(log_w))
        a_t = float(np.dot(w, alphas))
        out.append(float(np.quantile(residuals, np.clip(1 - a_t, 0, 1), method="inverted_cdf")))
        if len(residuals) > math.floor(1 / (1 - coverage)):
            beta = np.mean(np.asarray(residuals) >= s)
            losses = _pinball(beta, alphas, 1 - coverage)
            wbar = log_w - eta * losses
            log_w = logsumexp([wbar, np.full(k, logsumexp(wbar))], b=[[1 - sigma], [sigma / k]], axis=0)
            log_w = log_w - logsumexp(log_w)
            err = alphas > beta
            alphas = np.clip(alphas + gammas * ((1 - coverage) - err), 0, 1)
        residuals.append(s)
    return np.asarray(out)


def _aci(calib, stream, alpha, gamma=0.005):
    """Adaptive Conformal Inference (Gibbs & Candes 2021): alpha_{t+1} = alpha_t + gamma (alpha - err_t).
    If alpha_t <= 0 the ACI set is the whole line; it is capped at the largest error seen and counted."""
    a_t = alpha
    residuals = list(np.abs(calib))
    out, n_cap = [], 0
    for s in np.abs(stream):
        level = 1 - a_t
        if level >= 1:
            q = float(np.max(residuals))
            n_cap += 1
        elif level <= 0:
            q = 0.0
        else:
            q = float(np.quantile(residuals, level, method="inverted_cdf"))
        out.append(q)
        a_t = a_t + gamma * (alpha - float(s > q))
        residuals.append(s)
    return np.asarray(out), n_cap


# ---------------------------------------------------------------- N9: Extreme CP (ExtremeConformal GPD_max)
def _gpd_quantile(p, p0, u, sigma, xi):
    """ExtremeConformal GPD_quantiles: q = u + (sigma/xi) [((1-p)/(1-p0))^(-xi) - 1]  (xi = 0: log form)."""
    if abs(xi) < 1e-12:
        return u + sigma * np.log((1 - p0) / (1 - p))
    return u + (sigma / xi) * (((1 - p) / (1 - p0)) ** (-xi) - 1)


def _conformalize_gpd_max(s_cal, coverage, threshold_lvls=None, min_obs_gpd=10):
    """ExtremeConformal conformalize_GPD_max: for each threshold level p0 (default 0.80..0.99, capped so
    at least `min_obs_gpd` scores lie above it) fit a GPD to the exceedances over u = quantile(p0),
    compute the extrapolated `coverage` quantile, and return the maximum over thresholds."""
    s_cal = np.asarray(s_cal, dtype=float).ravel()
    n = len(s_cal)
    lv = np.round(np.arange(0.80, 0.9901, 0.01), 2) if threshold_lvls is None else np.asarray(threshold_lvls, float)
    if min_obs_gpd > 0:
        lv = np.minimum(lv, 1 - min_obs_gpd / n)
    lv = np.unique(lv)
    rows = []
    for p0 in lv:
        u = float(np.quantile(s_cal, p0))
        exc = s_cal[s_cal > u] - u
        try:
            xi, _, sigma = genpareto.fit(exc, floc=0)
            dq = float(_gpd_quantile(coverage, p0, u, sigma, xi))
        except Exception:
            xi = sigma = dq = np.nan
        rows.append({"threshold_level": float(p0), "threshold_u": u, "n_exceedances": int(len(exc)),
                     "sigma": float(sigma), "xi": float(xi), "q_threshold": dq})
    table = pd.DataFrame(rows)
    finite = table["q_threshold"].replace([np.inf, -np.inf], np.nan).dropna()
    return (float(finite.max()) if len(finite) else np.nan), table


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_NEXCP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    strategies_names = {
        "NexCP-Split": "NexCP Split",
        "NexCP-Full": "NexCP Full",
        "NexCP-J+aB": "NexCP Jackknife+ aB",
        "NexCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, SEED)

    # NexCP - Split, eq. (11)
    q, notes["NexCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NexCP-Split"] = yhat_test
    int_pred["NexCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NexCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NexCP-Full"] = yhat_test
    int_pred["NexCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NexCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-J+aB"] = bag_mean
    int_pred["NexCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NexCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NexCP-CV+"] = np.c_[lo, hi]

    # =========================================================
    # Metrics
    # =========================================================
    def coverage(y, l, u):
        return np.mean((y >= l) & (y <= u))

    def width(l, u):
        return np.mean(u - l)

    def cwc(l, u, y, alpha):
        cov = coverage(y, l, u)
        w = width(l, u)
        penalty = np.exp(-50 * (cov - (1 - alpha)))
        return w * (1 + penalty if cov < (1 - alpha) else 1)

    y_true = y_test.to_numpy()

    summary_rows = []

    print("\n===== NEXCP RESULTS =====")

    for s in strategies_names:
        cov = coverage(y_true, int_pred[s][:, 0], int_pred[s][:, 1])
        wid = width(int_pred[s][:, 0], int_pred[s][:, 1])
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc_val = cwc(int_pred[s][:, 0], int_pred[s][:, 1], y_true, alpha)

        summary_rows.append([
            strategies_names[s],
            cov,
            wid,
            rmse,
            cwc_val,
        ])

        print(f"{strategies_names[s]} - Coverage: {cov*100:.3f}% | Width: {wid:.3f}")

    summary_df = pd.DataFrame(
        summary_rows,
        columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"],
    )
    summary_df["Note"] = [notes.get(s, "") for s in strategies_names]  # FIX N10: caps / limits per method

    # =========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # =========================================================
    idx = np.arange(len(y_test))
    plot_buffers = {}

    for s in strategies_names:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_test, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(
            idx,
            int_pred[s][:, 0],
            int_pred[s][:, 1],
            color="gray",
            alpha=0.5,
            label="Prediction interval",
        )
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(strategies_names[s])
        ax.set_xlabel("Sample Number")
        ax.set_ylabel("str")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # Excel Export
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary sheet ----
        sheet_name = "NEXCP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name, 0)
        for r_idx, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        # ---- Plot sheets ----
        for s, buf in plot_buffers.items():
            img = Image(buf)

            sheet_name = f"NEXCP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1

            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values sheet ----
        results_dict = {}
        for s in strategies_names:
            results_dict[f"{s}_true"] = y_true
            results_dict[f"{s}_pred"] = y_pred[s]
            results_dict[f"{s}_lower"] = int_pred[s][:, 0]
            results_dict[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(results_dict)

        sheet_name = "NEXCP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r_idx, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_online_super(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30,
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    STRATEGY_NAMES = {
        "Online-Split": "Online Split",
        "Online-CV+": "Online CV+",
        "Online-J+aB": "Online Jackknife+ aB",
        "SAOCP": "SAOCP",
        "SF-OGD": "SF-OGD",
        "FACI": "FACI",
        "ACI": "ACI",   # FIX N5: added (Gibbs & Candes 2021, gamma = 0.005)
        "MFCS-Split": "MFCS Split",
        "MFCS-Full": "MFCS Full",
    }

    ALL_STRATEGIES = list(STRATEGY_NAMES.keys())
    if strategies == "all":
        strategies = ALL_STRATEGIES

    # =========================================================
    # FIX N1, N4-N6: corrected online methods. Every method starts from OUT-OF-FOLD training residuals;
    # test points arrive one at a time and each interval is OUTPUT before that point's error is used.
    # Was: in-sample training residuals; SF-OGD with flipped gradient sign and look-ahead; "FACI" and
    # "SAOCP" were window heuristics (FACI also looked ahead); J+aB scored models on their own samples.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    y_te = np.asarray(y_test).ravel()
    n_train, n_test = len(y_tr), len(y_te)

    base_model = model_class(**model_params)
    base_model.fit(X_tr, y_tr)
    yhat_te = np.asarray(base_model.predict(X_te)).ravel()
    res_te = np.abs(y_te - yhat_te)
    R_oof, mu_te_folds, _ = _oof_fold_predictions(model_class, model_params, X_tr, y_tr, X_te, 5, SEED)

    y_pred, int_pred, notes = {}, {}, {}

    def _band(name, centre, q):
        y_pred[name] = np.asarray(centre)
        int_pred[name] = np.c_[centre - q, centre + q]

    if "Online-Split" in strategies:
        q, notes["Online-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("Online-Split", yhat_te, q)

    if "Online-CV+" in strategies:
        # CV+ (Barber et al. 2021); past test points join the calibration set once their error is known
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_te_folds[:, j], np.full(j, yhat_te[j])]
            R = np.r_[R_oof, res_te[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-CV+"] = yhat_te
        int_pred["Online-CV+"] = np.c_[lo, hi]
        notes["Online-CV+"] = _cap_count_note(n_cap, n_test)

    if "Online-J+aB" in strategies:
        # J+aB (Kim, Xu & Barber 2020): out-of-bag residuals; past test points join as above
        R_oob, mu_oob, bag_mean, has = _jab_oob(model_class, model_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
        res_bag = np.abs(y_te - bag_mean)
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_oob[:, j], np.full(j, bag_mean[j])]
            R = np.r_[R_oob, res_bag[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-J+aB"] = bag_mean
        int_pred["Online-J+aB"] = np.c_[lo, hi]
        notes["Online-J+aB"] = _cap_count_note(n_cap, n_test)

    if "SAOCP" in strategies:
        _band("SAOCP", yhat_te, _saocp(R_oof, res_te, alpha))

    if "SF-OGD" in strategies:
        _band("SF-OGD", yhat_te, _sf_ogd(R_oof, res_te, alpha))

    if "FACI" in strategies:
        _band("FACI", yhat_te, _faci(R_oof, res_te, alpha))

    if "ACI" in strategies:
        q_seq, n_cap = _aci(R_oof, res_te, alpha, gamma=0.005)
        _band("ACI", yhat_te, q_seq)
        notes["ACI"] = "" if n_cap == 0 else f"{n_cap} of {n_test} steps had alpha_t <= 0 (infinite ACI set, capped at the largest error)"

    if "MFCS-Split" in strategies:
        # kept as written by the user (no published source found); same computation as Online-Split
        q, notes["MFCS-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("MFCS-Split", yhat_te, q)

    if "MFCS-Full" in strategies:
        # refit on all points seen so far; margin from that refit's OUT-OF-FOLD residuals (FIX N1)
        lo, hi, preds, n_cap = [], [], [], 0
        for j in range(n_test):
            X_h = np.vstack([X_tr, X_te[:j]])
            y_h = np.r_[y_tr, y_te[:j]]
            m = model_class(**model_params)
            m.fit(X_h, y_h)
            yhat = float(np.asarray(m.predict(X_te[j:j + 1])).ravel()[0])
            R_h, _, _ = _oof_fold_predictions(model_class, model_params, X_h, y_h, X_te[j:j + 1], 5, SEED)
            q = weighted_quantile(R_h, np.ones(len(R_h)), 1 - alpha)
            if not np.isfinite(q):
                q, n_cap = float(np.max(R_h)), n_cap + 1
            preds.append(yhat); lo.append(yhat - q); hi.append(yhat + q)
        y_pred["MFCS-Full"] = np.asarray(preds)
        int_pred["MFCS-Full"] = np.c_[lo, hi]
        notes["MFCS-Full"] = _cap_count_note(n_cap, n_test)

    # =========================================================
    # METRICS
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rows = []
    for s in y_pred:
        ymin, ymax = int_pred[s].T
        cov = np.mean((y_true >= ymin) & (y_true <= ymax))
        wid = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc = wid * (1 + (cov < (1 - alpha)) * (1 - cov))

        rows.append([STRATEGY_NAMES[s], cov, wid, rmse, cwc])

    summary_df = pd.DataFrame(
        rows, columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"]
    )
    summary_df["Note"] = [notes.get(s, "") for s in y_pred]  # FIX N10: caps / limits per method

    print("\n===== ONLINE CP RESULTS =====")
    print(summary_df)

    # =========================================================
    # PLOTTING (SAVE ONLY)
    # =========================================================
    plot_buffers = {}
    idx = np.arange(len(y_test))

    for s in y_pred:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_true, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(idx,
                        int_pred[s][:, 0],
                        int_pred[s][:, 1],
                        color="gray",
                        alpha=0.5,
                        label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(f"{STRATEGY_NAMES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel("str")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # EXCEL EXPORT (APPEND SAFE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "ONLINECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plots ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"ONLINECP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values ----
        values = {}
        for s in y_pred:
            values[f"{s}_true"] = y_true
            values[f"{s}_pred"] = y_pred[s]
            values[f"{s}_lower"] = int_pred[s][:, 0]
            values[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(values)

        sheet_name = "ONLINECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_COP(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lr=0.05,          # Learning rate for COP
    T_burnin=50,      # Window size for distribution estimation
    scale=0.5,        # Scale factor for the integrator
    random_state=123,
):
    """
    Conformal Optimistic Prediction (COP)
    ✔ Implements the Distribution-Informed Gradient logic from the repository.
    ✔ Matches the "Excel-safe" and plotting structure of your previous functions.
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # 1. Train Base Model & Generate Test Predictions
    # =========================================================
    # COP is an online calibration method, but we usually start with a pre-trained model
    # or an online model. Here we fit on Train and calibrate on Test stream.
    model = model_class(**model_params)
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred_test = model.predict(X_test).ravel()
    y_true_test = y_test.to_numpy().ravel()

    # Calculate non-conformity scores (Absolute Residuals)
    # This assumes symmetric intervals.
    scores = np.abs(y_true_test - y_pred_test)
    T_test = len(scores)

    # =========================================================
    # 2. COP Initialization
    # =========================================================
    # Initialize quantiles based on training data residuals if possible
    y_pred_train = model.predict(X_train).ravel()
    scores_train = np.abs(y_train.to_numpy().ravel() - y_pred_train)
    # FIX N1: start from OUT-OF-FOLD training residuals (was: in-sample residuals, too small)
    R_oof_cop, _, _ = _oof_fold_predictions(model_class, model_params, np.asarray(X_train),
                                            np.asarray(y_train).ravel(), np.asarray(X_test)[:1], 5, 123)
    init_q = np.quantile(R_oof_cop, 1 - alpha)

    # Arrays to store history
    qs = np.zeros(T_test)          # Final adaptive quantile
    qts = np.zeros(T_test)         # Base quantile term
    integrators = np.zeros(T_test) # Optimistic integrator term
    covereds = np.zeros(T_test)    # Coverage boolean

    # Set initial values
    qs[0] = init_q
    qts[0] = init_q

    # =========================================================
    # 3. Online COP Loop
    # =========================================================
    # Logic adapted from `core/methods.py` function `COP`

    for t in range(T_test - 1):
        # 1. Observe coverage at current step t
        covereds[t] = 1 if qs[t] >= scores[t] else 0

        # FIX N8: step size proportional to the score scale, as in the official COP code
        # (lr x (max - min) of the last T_burnin scores). Was: a fixed lr = 0.05 in target units.
        _hist = np.r_[R_oof_cop, scores[:t + 1]][-T_burnin:]
        lr_t = lr * (np.max(_hist) - np.min(_hist))

        # 2. Calculate Standard Gradient
        # If covered, we decrease q (grad is positive). If not, increase q.
        grad = alpha if covereds[t] else -(1 - alpha)

        # 3. Calculate Distribution-Informed Gradient (The "Optimistic" part)
        if t < T_burnin:
            grad_i = 0
        else:
            # Look at the window of recent scores
            window_s = scores[t - T_burnin : t]

            # Calculate ECDF at the predicted next quantile step
            # logic: np.mean(window_s <= qts[t] - lr * grad) - (1 - alpha)
            current_target = qts[t] - lr_t * grad   # FIX N8
            grad_i = np.mean(window_s <= current_target) - (1 - alpha)

        # 4. Update Terms
        integrator = -scale * grad_i

        # Update base quantile (Gradient Descent)
        qts[t + 1] = qts[t] - lr_t * grad   # FIX N8

        # Update integrator state
        integrators[t + 1] = lr_t * integrator   # FIX N8

        # Combine for final quantile
        qs[t + 1] = qts[t + 1] + integrators[t + 1]

        # Safety: Ensure quantile implies non-negative width
        qs[t + 1] = max(0, qs[t + 1])

    # Handle the final step coverage for metrics
    covereds[-1] = 1 if qs[-1] >= scores[-1] else 0

    # =========================================================
    # 4. Construct Prediction Intervals
    # =========================================================
    ymin = y_pred_test - qs
    ymax = y_pred_test + qs

    # =========================================================
    # 5. Metrics
    # =========================================================
    rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    coverage = np.mean(covereds)
    avg_width = np.mean(ymax - ymin)

    print(f"\n===== CONFORMAL OPTIMISTIC PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Avg Width: {avg_width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"Final q  : {qs[-1]:.3f}")

    # =========================================================
    # 6. Summary Tables
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "COP",
        "Coverage": coverage,
        "Width": avg_width,
        "RMSE": rmse,
        "Alpha": alpha,
        "LR": lr,
        "Scale": scale
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true": y_true_test,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": ymax - ymin,
        "is_covered": covereds.astype(int),
        "score_residual": scores,
        "adaptive_q": qs
    })

    # =========================================================
    # 7. Plotting
    # =========================================================
    idx = np.arange(len(y_true_test))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax.fill_between(idx, ymin, ymax, color="gray", alpha=0.5, label="COP Interval")
    ax.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")

    ax.set_title(f"Conformal Optimistic Prediction (COP) - {model_name}")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel("str")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # 8. Excel Export
    # =========================================================
    if excel_file_path:
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            # Create a new workbook if it doesn't exist
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active) # Remove default sheet

        # ---- Summary ----
        sheet_name = "COP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "COP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "COP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_EXTREME(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    calib_size=0.2,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30,
    random_state=123,
):
    """
    Extreme Conformal Prediction (upper-tail, single-sided)
    ✔ Same structure as NexCP & OnlineCP
    ✔ No return values
    ✔ Excel-safe
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # Train / calibration split
    # =========================================================
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train,
        y_train,
        test_size=calib_size,
        random_state=random_state,
        shuffle=True,
    )

    # =========================================================
    # Train model
    # =========================================================
    model = model_class(**model_params)
    model.fit(X_fit, y_fit)

    y_pred_test = model.predict(X_test).ravel()
    y_pred_calib = model.predict(X_calib).ravel()

    # =========================================================
    # Nonconformity scores (upper tail only)
    # =========================================================
    # FIX N9: Extreme CP as in the official ExtremeConformal code (conformalize_GPD_max):
    # signed scores y - y_hat on ALL calibration points, thresholds 0.80..0.99 (capped so >= 10 scores
    # exceed them), official GPD quantile formula, maximum over thresholds; ONE-SIDED upper bound.
    # Was: only positive scores kept, (1/alpha)^xi in the formula (missing the threshold share), a hard
    # error with < 30 exceedances, and a two-sided band reported for a one-sided method.
    scores = np.asarray(y_calib).ravel() - y_pred_calib
    q_extreme, gpd_table = _conformalize_gpd_max(scores, 1 - alpha, min_obs_gpd=10)
    extreme_note = ""
    if not np.isfinite(q_extreme):
        q_extreme = weighted_quantile(scores, np.ones(len(scores)), 1 - alpha)
        extreme_note = "GPD fit failed at every threshold; empirical conformal quantile used"
    best_row = gpd_table.loc[gpd_table["q_threshold"].idxmax()] if gpd_table["q_threshold"].notna().any() else None
    u = float(best_row["threshold_u"]) if best_row is not None else np.nan
    n_exc_used = int(best_row["n_exceedances"]) if best_row is not None else 0
    print(gpd_table.to_string(index=False))

    # one-sided interval (-inf, y_hat + q]
    y_pred_test = np.asarray(y_pred_test).ravel()
    ymin = np.array([None] * len(y_pred_test), dtype=object)   # no lower bound (empty cell in Excel)
    ymax = y_pred_test + q_extreme

    # =========================================================
    # Metrics
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred_test))
    coverage = np.mean(y_true <= ymax)
    width = q_extreme  # meaningful one-sided width

    print(f"\n===== EXTREME CONFORMAL PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Width    : {width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"q_extreme: {q_extreme:.3f}")
    print(f"u        : {u:.3f}")
    print(f"#Exceed. : {n_exc_used}")
    if extreme_note:
        print(f"Note     : {extreme_note}")

    # =========================================================
    # Summary table
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "EXTREME CP",
        "Coverage": coverage,
        "Width": width,
        "RMSE": rmse,
        "q_extreme": q_extreme,
        "threshold_u": u,
        "n_exceedances": n_exc_used,
        "Note": extreme_note,   # FIX N10
    }])

    # =========================================================
    # Values table
    # =========================================================
    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": np.full(len(y_pred_test), q_extreme),   # FIX N9: one-sided width = q
        "is_covered": (y_true <= ymax).astype(int),
        "residual": y_true - y_pred_test,
    })

    # =========================================================
    # Plot (SAVE ONLY)
    # =========================================================
    idx = np.arange(len(y_true))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(idx, y_true, s=10, alpha=0.7, label="True")
    ax.plot(idx, ymax, ls="--", lw=1, label="Upper bound (one-sided)")   # FIX N9
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Prediction")

    ax.set_title(f"Extreme Conformal Prediction ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel("str")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # Excel export (append-safe, SAME FILE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "EXTREMECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "EXTREMECP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "EXTREMECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # FIX N9: GPD fit at every threshold level (the reported q is the maximum)
        sheet_name = "EXTREMECP_GPD_Thresholds"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        _tbl = gpd_table.astype(object).where(gpd_table.notna(), None)
        for r, row in enumerate(dataframe_to_rows(_tbl, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


In [18]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path



import os
try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("[Info] Install tqdm for progress bars: pip install tqdm")


# ═══════════════════════════════════════════════════════════
# AlphaNet Architecture  (FIXED)
# ═══════════════════════════════════════════════════════════
class AlphaNet(nn.Module):
    """
    Maps a feature vector (sum of LOO scores) → learned α̃.
    Output is clamped to (alpha_min, 1) so that the conformal
    denominator α̃·(n+1)−1 is always positive.
    """

    def __init__(self, input_dim, alpha_min=0.05):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.alpha_min = alpha_min  # lower bound for valid α̃
        self._initialize_near_one()

    def _initialize_near_one(self):
        """Bias output toward α̃ ≈ 1 at init (sigmoid(5) ≈ 0.993)."""
        with torch.no_grad():
            self.fc2.weight.data.normal_(0, 0.01)
            self.fc2.bias.data.fill_(5.0)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        raw = self.sigmoid(self.fc2(x)).squeeze(-1)
        # Remap from (0,1) → (alpha_min, 1) so denominator stays positive
        return self.alpha_min + (1.0 - self.alpha_min) * raw


def _interval_size(scores, alpha, eps=1e-8):
    """
    Conformal interval width = 2 * Σscores / (α̃·(n+1) − 1).
    Denominator is clamped to be strictly positive.
    """
    n = len(scores)
    denom = alpha * (n + 1) - 1
    denom = torch.clamp(denom, min=eps)   # ← KEY FIX: prevent negative/zero
    r = scores.sum() / denom
    return 2 * r


def _fmt_time(seconds):
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}min"
    else:
        return f"{seconds/3600:.1f}hr"


# ═══════════════════════════════════════════════════════════
# Main function  (CORRECTED + PROGRESS TRACKING)
# ═══════════════════════════════════════════════════════════
def conformal_predictions_AlphaNet(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    """
    Learned-α Conformal Prediction via AlphaNet.
    - FIXED: α̃ clamped to valid range (no negative intervals)
    - FIXED: LOO scores precomputed (massive speedup)
    - ADDED: tqdm progress bars + ETA estimates
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # --- numpy-ify if pandas ---
    y_train_np = y_train.to_numpy().ravel() if hasattr(y_train, "to_numpy") else np.asarray(y_train).ravel()
    y_test_np  = y_test.to_numpy().ravel()  if hasattr(y_test,  "to_numpy") else np.asarray(y_test).ravel()
    X_train_np = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    X_test_np  = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)

    # =================================================================
    # 1.  Split training data → fit + calibration
    # =================================================================
    n_total = len(X_train_np)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = X_train_np[fit_idx],   y_train_np[fit_idx]
    X_calib, y_calib = X_train_np[calib_idx],  y_train_np[calib_idx]
    n_calib = len(X_calib)

    # Compute alpha_min: must satisfy α̃·(n+1) − 1 > 0  →  α̃ > 1/(n+1)
    alpha_min = 1.0 / (n_calib + 1) + 0.01   # small margin above threshold
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =================================================================
    # 2.  Train base regression model
    # =================================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**model_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    # =================================================================
    # 3.  PRECOMPUTE all LOO scores  (KEY SPEEDUP)
    # =================================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = tqdm(range(n_calib), desc="    LOO precompute", leave=False) if HAS_TQDM else range(n_calib)

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        scores_loo = np.abs(base_model.predict(loo_X).ravel() - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # Build feature matrix (sum of LOO scores per sample)
    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =================================================================
    # 4.  Train AlphaNet (multiple runs, keep best)
    # =================================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net   = None
    best_loss  = np.inf
    history    = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    # USE PRECOMPUTED scores instead of calling predict()
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =================================================================
    # 5.  Compute conformal intervals on the TEST set
    # =================================================================
    print(f"  [Step 5/9] Computing conformal intervals on test set...")

    scores_calib = np.abs(base_model.predict(X_calib).ravel() - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small (denom={denom:.4f}); fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = base_model.predict(X_test_np).ravel()
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =================================================================
    # 6.  Metrics
    # =================================================================
    y_true_test = y_test_np
    is_covered  = ((y_true_test >= ymin) & (y_true_test <= ymax)).astype(int)
    coverage    = is_covered.mean()
    avg_width   = np.mean(ymax - ymin)
    rmse        = np.sqrt(mean_squared_error(y_true_test, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =================================================================
    # 7.  Summary & values DataFrames
    # =================================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true":       y_true_test,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =================================================================
    # 8.  Plotting (SAVE ONLY — NO SHOW)
    # =================================================================
    print(f"  [Step 8/9] Generating plots...")
    idx = np.arange(len(y_true_test))

    # --- PI plot ---
    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal – {model_name}")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel("str")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    # --- Training curves ---
    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =================================================================
    # 9.  Excel Export
    # =================================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        sn = _unique_sheet(wb, "AlphaNet_Summary")
        ws = wb.create_sheet(sn, 0)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        sn = _unique_sheet(wb, "AlphaNet_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_pi), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_tc), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet Conformal complete in {_fmt_time(total_time)}")
    return summary_df

# **Tuned Parameters**

In [19]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/pile_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/pile-tuned-params-v4/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

In [21]:
conformal_predictions_NEXCP(
    model_class=LGBMRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full LightGBM: 5/24 test points, 0.4 min


  NexCP-Full LightGBM: 10/24 test points, 0.8 min


  NexCP-Full LightGBM: 15/24 test points, 1.1 min


  NexCP-Full LightGBM: 20/24 test points, 1.4 min


  NexCP-Full LightGBM: 24/24 test points, 1.7 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 91.667% | Width: 2.315
NexCP Full - Coverage: 83.333% | Width: 3.410
NexCP Jackknife+ aB - Coverage: 91.667% | Width: 2.153
NexCP CV+ - Coverage: 91.667% | Width: 2.460


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.916667,2.315487,0.760748,2.315487,
1,NexCP Full,0.833333,3.410224,0.760748,99.004352,17 of 24 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.916667,2.152875,0.564509,2.152875,
3,NexCP CV+,0.916667,2.460377,0.622768,2.460377,


In [22]:
conformal_predictions_online_super(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.791667  1.982626  0.760748  2.395673     
1            Online CV+  0.916667  2.367286  0.760748  2.367286     
2  Online Jackknife+ aB  0.916667  2.078874  0.567048  2.078874     
3                 SAOCP  0.875000  2.424516  0.760748  2.727581     
4                SF-OGD  0.916667  2.524648  0.760748  2.524648     
5                  FACI  0.833333  2.238186  0.760748  2.611217     
6                   ACI  0.875000  2.150078  0.760748  2.418838     
7            MFCS Split  0.791667  1.982626  0.760748  2.395673     
8             MFCS Full  0.916667  2.367650  0.573617  2.367650     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.791667,1.982626,0.760748,2.395673,
1,Online CV+,0.916667,2.367286,0.760748,2.367286,
2,Online Jackknife+ aB,0.916667,2.078874,0.567048,2.078874,
3,SAOCP,0.875000,2.424516,0.760748,2.727581,
4,SF-OGD,0.916667,2.524648,0.760748,2.524648,
5,FACI,0.833333,2.238186,0.760748,2.611217,
6,ACI,0.875000,2.150078,0.760748,2.418838,
7,MFCS Split,0.791667,1.982626,0.760748,2.395673,
8,MFCS Full,0.916667,2.367650,0.573617,2.367650,


In [23]:
conformal_predictions_COP(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (LightGBM) =====
Coverage : 83.333%
Avg Width: 2.015
RMSE     : 0.761
Final q  : 0.991


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.833333,2.014581,0.760748,0.1,0.05,0.5


In [24]:
conformal_predictions_EXTREME(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
             0.6     0.007761             10 0.372193 -0.509826     0.377719

===== EXTREME CONFORMAL PREDICTION (LightGBM) =====
Coverage : 70.833%
Width    : 0.378
RMSE     : 0.504
q_extreme: 0.378
u        : 0.008
#Exceed. : 10


In [25]:
conformal_predictions_AlphaNet(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base LightGBM model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.7894 | Size: 0.9237 | α̃: 0.9933 | ⏱ 0.36s/epoch | ETA: 33.9min


  λ=20.0 | Run 1/4 | Epoch 100/1000 | Loss: 8.9589 | Size: 4.2795 | α̃: 0.2340 | ⏱ 0.01s/epoch | ETA: 59s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 8.9325 | Size: 4.1852 | α̃: 0.2374 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 8.9325
  [Step 4/9] Training done in 41s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (LightGBM)
  Learned α̃     : 0.2333
  Half-width (r) : 2.1317
  Coverage       : 100.000%
  Avg Width      : 4.2634
  RMSE           : 0.6383
  Total time     : 42s
    ├─ Precompute: 0s
    └─ Training  : 41s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 42s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.263397,0.638254,0.1,0.766742,0.0,0.233258,20.0,41.75


# **Conformal Predictions with XGBoost**

In [26]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

In [27]:
conformal_predictions_NEXCP(
    model_class=XGBRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full XGBoost: 5/24 test points, 0.9 min


  NexCP-Full XGBoost: 10/24 test points, 1.7 min


  NexCP-Full XGBoost: 15/24 test points, 2.6 min


  NexCP-Full XGBoost: 20/24 test points, 3.5 min


  NexCP-Full XGBoost: 24/24 test points, 4.1 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 91.667% | Width: 2.374
NexCP Full - Coverage: 87.500% | Width: 2.091
NexCP Jackknife+ aB - Coverage: 91.667% | Width: 2.277
NexCP CV+ - Coverage: 91.667% | Width: 2.517


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.916667,2.373784,0.714722,2.373784,
1,NexCP Full,0.875000,2.091191,0.714722,9.390166,1 of 24 intervals reach the edge of the y-grid...
2,NexCP Jackknife+ aB,0.916667,2.277004,0.575792,2.277004,
3,NexCP CV+,0.916667,2.516762,0.640292,2.516762,


In [28]:
conformal_predictions_online_super(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.916667  2.326519  0.714722  2.326519     
1            Online CV+  0.916667  2.430971  0.714722  2.430971     
2  Online Jackknife+ aB  0.916667  2.170365  0.594102  2.170365     
3                 SAOCP  0.875000  2.602976  0.714722  2.928348     
4                SF-OGD  0.875000  2.393164  0.714722  2.692309     
5                  FACI  0.916667  2.264995  0.714722  2.264995     
6                   ACI  0.916667  2.281798  0.714722  2.281798     
7            MFCS Split  0.916667  2.326519  0.714722  2.326519     
8             MFCS Full  0.916667  2.315937  0.532274  2.315937     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.916667,2.326519,0.714722,2.326519,
1,Online CV+,0.916667,2.430971,0.714722,2.430971,
2,Online Jackknife+ aB,0.916667,2.170365,0.594102,2.170365,
3,SAOCP,0.875000,2.602976,0.714722,2.928348,
4,SF-OGD,0.875000,2.393164,0.714722,2.692309,
5,FACI,0.916667,2.264995,0.714722,2.264995,
6,ACI,0.916667,2.281798,0.714722,2.281798,
7,MFCS Split,0.916667,2.326519,0.714722,2.326519,
8,MFCS Full,0.916667,2.315937,0.532274,2.315937,


In [29]:
conformal_predictions_COP(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (XGBoost) =====
Coverage : 91.667%
Avg Width: 2.257
RMSE     : 0.715
Final q  : 1.031


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.916667,2.257488,0.714722,0.1,0.05,0.5


In [30]:
conformal_predictions_EXTREME(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances   sigma       xi  q_threshold
             0.6    -0.117966             10 0.38986 0.013718     0.427666

===== EXTREME CONFORMAL PREDICTION (XGBoost) =====
Coverage : 75.000%
Width    : 0.428
RMSE     : 0.576
q_extreme: 0.428
u        : -0.118
#Exceed. : 10


In [31]:
conformal_predictions_AlphaNet(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base XGBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.8329 | Size: 0.9673 | α̃: 0.9933 | ⏱ 0.01s/epoch | ETA: 2.0min


  λ=20.0 | Run 1/4 | Epoch 100/1000 | Loss: 9.1695 | Size: 4.4104 | α̃: 0.2380 | ⏱ 0.01s/epoch | ETA: 39s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 9.1415 | Size: 4.2805 | α̃: 0.2431 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 9.1415
  [Step 4/9] Training done in 40s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (XGBoost)
  Learned α̃     : 0.2375
  Half-width (r) : 2.1923
  Coverage       : 100.000%
  Avg Width      : 4.3846
  RMSE           : 0.6376
  Total time     : 40s
    ├─ Precompute: 0s
    └─ Training  : 40s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 40s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.38463,0.63764,0.1,0.762485,0.0,0.237515,20.0,40.21


# **Conformal Predictions with GPBoost**

In [32]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

In [33]:
conformal_predictions_NEXCP(
    model_class=GPBoostRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full GPBoost: 5/24 test points, 0.2 min


  NexCP-Full GPBoost: 10/24 test points, 0.4 min


  NexCP-Full GPBoost: 15/24 test points, 0.6 min


  NexCP-Full GPBoost: 20/24 test points, 0.8 min


  NexCP-Full GPBoost: 24/24 test points, 1.0 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 91.667% | Width: 2.382
NexCP Full - Coverage: 75.000% | Width: 2.434
NexCP Jackknife+ aB - Coverage: 91.667% | Width: 2.141
NexCP CV+ - Coverage: 95.833% | Width: 2.607


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.916667,2.381714,0.674262,2.381714,
1,NexCP Full,0.750000,2.434371,0.674262,4403.880123,2 of 24 intervals reach the edge of the y-grid...
2,NexCP Jackknife+ aB,0.916667,2.141204,0.554221,2.141204,
3,NexCP CV+,0.958333,2.607103,0.580255,2.607103,


In [34]:
conformal_predictions_online_super(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.916667  2.236445  0.674262  2.236445     
1            Online CV+  0.916667  2.393326  0.674262  2.393326     
2  Online Jackknife+ aB  0.916667  2.165728  0.557574  2.165728     
3                 SAOCP  0.791667  2.096256  0.674262  2.532976     
4                SF-OGD  0.916667  2.791431  0.674262  2.791431     
5                  FACI  0.916667  2.123207  0.674262  2.123207     
6                   ACI  0.916667  2.142483  0.674262  2.142483     
7            MFCS Split  0.916667  2.236445  0.674262  2.236445     
8             MFCS Full  0.958333  2.479144  0.519237  2.479144     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.916667,2.236445,0.674262,2.236445,
1,Online CV+,0.916667,2.393326,0.674262,2.393326,
2,Online Jackknife+ aB,0.916667,2.165728,0.557574,2.165728,
3,SAOCP,0.791667,2.096256,0.674262,2.532976,
4,SF-OGD,0.916667,2.791431,0.674262,2.791431,
5,FACI,0.916667,2.123207,0.674262,2.123207,
6,ACI,0.916667,2.142483,0.674262,2.142483,
7,MFCS Split,0.916667,2.236445,0.674262,2.236445,
8,MFCS Full,0.958333,2.479144,0.519237,2.479144,


In [35]:
conformal_predictions_COP(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (GPBoost) =====
Coverage : 91.667%
Avg Width: 2.131
RMSE     : 0.674
Final q  : 0.936


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.916667,2.131077,0.674262,0.1,0.05,0.5


In [36]:
conformal_predictions_EXTREME(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma       xi  q_threshold
             0.6    -0.072541             10 1.099204 -1.28885     0.637456

===== EXTREME CONFORMAL PREDICTION (GPBoost) =====
Coverage : 79.167%
Width    : 0.637
RMSE     : 0.536
q_extreme: 0.637
u        : -0.073
#Exceed. : 10


In [37]:
conformal_predictions_AlphaNet(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base GPBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.9661 | Size: 1.1004 | α̃: 0.9933 | ⏱ 0.01s/epoch | ETA: 1.9min


  λ=20.0 | Run 1/4 | Epoch 100/1000 | Loss: 9.7283 | Size: 4.6511 | α̃: 0.2539 | ⏱ 0.01s/epoch | ETA: 37s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 9.7075 | Size: 4.5660 | α̃: 0.2571 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 9.7075
  [Step 4/9] Training done in 40s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (GPBoost)
  Learned α̃     : 0.2496
  Half-width (r) : 2.3572
  Coverage       : 100.000%
  Avg Width      : 4.7144
  RMSE           : 0.7029
  Total time     : 40s
    ├─ Precompute: 0s
    └─ Training  : 40s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 40s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.714418,0.702921,0.1,0.750444,0.0,0.249556,20.0,40.45


# **Conformal Predictions with NGBoost**

In [38]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

In [39]:
conformal_predictions_NEXCP(
    model_class=NGBRegressor,
    best_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full NGBoost: 5/24 test points, 18.1 min


  NexCP-Full NGBoost: 10/24 test points, 36.0 min


  NexCP-Full NGBoost: 15/24 test points, 54.9 min


  NexCP-Full NGBoost: 20/24 test points, 73.3 min


  NexCP-Full NGBoost: 24/24 test points, 87.9 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 95.833% | Width: 3.066
NexCP Full - Coverage: 95.833% | Width: 5.610
NexCP Jackknife+ aB - Coverage: 95.833% | Width: 2.500
NexCP CV+ - Coverage: 100.000% | Width: 2.928


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.958333,3.065550,0.717648,3.065550,
1,NexCP Full,0.958333,5.609748,0.717648,5.609748,21 of 24 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.958333,2.499833,0.493133,2.499833,
3,NexCP CV+,1.000000,2.928136,0.557856,2.928136,


In [40]:
conformal_predictions_online_super(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.916667  2.837424  0.717648  2.837424     
1            Online CV+  0.958333  2.724915  0.717648  2.724915     
2  Online Jackknife+ aB  0.958333  2.251934  0.510778  2.251934     
3                 SAOCP  0.875000  2.562525  0.717648  2.882841     
4                SF-OGD  0.875000  2.685462  0.717648  3.021144     
5                  FACI  0.875000  2.499099  0.717648  2.811486     
6                   ACI  0.875000  2.518871  0.717648  2.833730     
7            MFCS Split  0.916667  2.837424  0.717648  2.837424     
8             MFCS Full  1.000000  2.503404  0.488162  2.503404     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.916667,2.837424,0.717648,2.837424,
1,Online CV+,0.958333,2.724915,0.717648,2.724915,
2,Online Jackknife+ aB,0.958333,2.251934,0.510778,2.251934,
3,SAOCP,0.875000,2.562525,0.717648,2.882841,
4,SF-OGD,0.875000,2.685462,0.717648,3.021144,
5,FACI,0.875000,2.499099,0.717648,2.811486,
6,ACI,0.875000,2.518871,0.717648,2.833730,
7,MFCS Split,0.916667,2.837424,0.717648,2.837424,
8,MFCS Full,1.000000,2.503404,0.488162,2.503404,


In [41]:
conformal_predictions_COP(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (NGBoost) =====
Coverage : 87.500%
Avg Width: 2.384
RMSE     : 0.718
Final q  : 1.171


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.875,2.383728,0.717648,0.1,0.05,0.5


In [42]:
conformal_predictions_EXTREME(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
             0.6    -0.071883             10 0.723327 -1.150887     0.429145

===== EXTREME CONFORMAL PREDICTION (NGBoost) =====
Coverage : 66.667%
Width    : 0.429
RMSE     : 0.686
q_extreme: 0.429
u        : -0.072
#Exceed. : 10


In [43]:
conformal_predictions_AlphaNet(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base NGBoost model...


  [Step 2/9] Done in 4s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 8s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.8169 | Size: 0.9513 | α̃: 0.9933 | ⏱ 0.02s/epoch | ETA: 2.5min


... (46 outputs left out)


  ALPHANET CONFORMAL (NGBoost)
  Learned α̃     : 0.2390
  Half-width (r) : 2.1390
  Coverage       : 100.000%
  Avg Width      : 4.2780
  RMSE           : 0.5905
  Total time     : 51s
    ├─ Precompute: 8s
    └─ Training  : 39s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 51s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.277985,0.590462,0.1,0.761002,0.0,0.238998,20.0,51.16


# **Conformal Predictions with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

In [45]:
conformal_predictions_NEXCP(
    model_class=GradientBoostingRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full GradientBoosting: 5/24 test points, 3.7 min


  NexCP-Full GradientBoosting: 10/24 test points, 6.1 min


  NexCP-Full GradientBoosting: 15/24 test points, 8.5 min


  NexCP-Full GradientBoosting: 20/24 test points, 10.9 min


  NexCP-Full GradientBoosting: 24/24 test points, 12.7 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 87.500% | Width: 2.573
NexCP Full - Coverage: 100.000% | Width: 5.141
NexCP Jackknife+ aB - Coverage: 95.833% | Width: 2.335
NexCP CV+ - Coverage: 100.000% | Width: 2.661


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.875000,2.572679,0.681943,11.552212,
1,NexCP Full,1.000000,5.140983,0.681943,5.140983,24 of 24 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.958333,2.334535,0.468326,2.334535,
3,NexCP CV+,1.000000,2.661219,0.499716,2.661219,


In [46]:
conformal_predictions_online_super(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.833333  2.443070  0.681943  2.850248     
1            Online CV+  1.000000  2.580637  0.681943  2.580637     
2  Online Jackknife+ aB  0.958333  1.988444  0.477662  1.988444     
3                 SAOCP  0.875000  2.802986  0.681943  3.153359     
4                SF-OGD  0.875000  2.760010  0.681943  3.105011     
5                  FACI  0.875000  2.525583  0.681943  2.841281     
6                   ACI  0.833333  2.450424  0.681943  2.858828     
7            MFCS Split  0.833333  2.443070  0.681943  2.850248     
8             MFCS Full  1.000000  2.431814  0.388692  2.431814     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.833333,2.443070,0.681943,2.850248,
1,Online CV+,1.000000,2.580637,0.681943,2.580637,
2,Online Jackknife+ aB,0.958333,1.988444,0.477662,1.988444,
3,SAOCP,0.875000,2.802986,0.681943,3.153359,
4,SF-OGD,0.875000,2.760010,0.681943,3.105011,
5,FACI,0.875000,2.525583,0.681943,2.841281,
6,ACI,0.833333,2.450424,0.681943,2.858828,
7,MFCS Split,0.833333,2.443070,0.681943,2.850248,
8,MFCS Full,1.000000,2.431814,0.388692,2.431814,


In [47]:
conformal_predictions_COP(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (GradientBoosting) =====
Coverage : 83.333%
Avg Width: 2.334
RMSE     : 0.682
Final q  : 1.191


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.833333,2.33433,0.681943,0.1,0.05,0.5


In [48]:
conformal_predictions_EXTREME(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
             0.6    -0.328005             10 0.622674 -0.039847     0.511796

===== EXTREME CONFORMAL PREDICTION (GradientBoosting) =====
Coverage : 83.333%
Width    : 0.512
RMSE     : 0.707
q_extreme: 0.512
u        : -0.328
#Exceed. : 10


In [49]:
conformal_predictions_AlphaNet(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base GradientBoosting model...


  [Step 2/9] Done in 1s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.8141 | Size: 0.9484 | α̃: 0.9933 | ⏱ 0.03s/epoch | ETA: 3.7min


... (45 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 9.0613 | Size: 4.2516 | α̃: 0.2405 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 9.0613
  [Step 4/9] Training done in 38s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (GradientBoosting)
  Learned α̃     : 0.2379
  Half-width (r) : 2.1464
  Coverage       : 100.000%
  Avg Width      : 4.2929
  RMSE           : 0.5907
  Total time     : 38s
    ├─ Precompute: 0s
    └─ Training  : 38s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 38s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.29287,0.590714,0.1,0.762054,0.0,0.237946,20.0,38.34


# **Conformal Predictions with CatBoost**

In [50]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

In [51]:
conformal_predictions_NEXCP(
    model_class=CatBoostRegressor,
    best_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full CatBoost: 5/24 test points, 1.1 min


  NexCP-Full CatBoost: 10/24 test points, 2.1 min


  NexCP-Full CatBoost: 15/24 test points, 3.1 min


  NexCP-Full CatBoost: 20/24 test points, 4.2 min


  NexCP-Full CatBoost: 24/24 test points, 5.1 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 95.833% | Width: 2.247
NexCP Full - Coverage: 87.500% | Width: 2.316
NexCP Jackknife+ aB - Coverage: 100.000% | Width: 2.362
NexCP CV+ - Coverage: 95.833% | Width: 2.281


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.958333,2.247457,0.505982,2.247457,
1,NexCP Full,0.875000,2.316257,0.505982,10.400787,11 of 24 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,1.000000,2.361612,0.473628,2.361612,
3,NexCP CV+,0.958333,2.281328,0.474077,2.281328,


In [52]:
conformal_predictions_online_super(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.958333  2.086879  0.505982  2.086879     
1            Online CV+  0.958333  2.115573  0.505982  2.115573     
2  Online Jackknife+ aB  0.958333  2.315183  0.474716  2.315183     
3                 SAOCP  0.875000  1.786380  0.505982  2.009678     
4                SF-OGD  0.916667  2.589944  0.505982  2.589944     
5                  FACI  0.916667  1.940899  0.505982  1.940899     
6                   ACI  0.958333  2.010809  0.505982  2.010809     
7            MFCS Split  0.958333  2.086879  0.505982  2.086879     
8             MFCS Full  0.958333  2.268243  0.469311  2.268243     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.958333,2.086879,0.505982,2.086879,
1,Online CV+,0.958333,2.115573,0.505982,2.115573,
2,Online Jackknife+ aB,0.958333,2.315183,0.474716,2.315183,
3,SAOCP,0.875000,1.786380,0.505982,2.009678,
4,SF-OGD,0.916667,2.589944,0.505982,2.589944,
5,FACI,0.916667,1.940899,0.505982,1.940899,
6,ACI,0.958333,2.010809,0.505982,2.010809,
7,MFCS Split,0.958333,2.086879,0.505982,2.086879,
8,MFCS Full,0.958333,2.268243,0.469311,2.268243,


In [53]:
conformal_predictions_COP(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (CatBoost) =====
Coverage : 95.833%
Avg Width: 1.820
RMSE     : 0.506
Final q  : 0.794


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.958333,1.819548,0.505982,0.1,0.05,0.5


In [54]:
conformal_predictions_EXTREME(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
             0.6    -0.098614             10 0.348427 -0.027714     0.375247

===== EXTREME CONFORMAL PREDICTION (CatBoost) =====
Coverage : 75.000%
Width    : 0.375
RMSE     : 0.508
q_extreme: 0.375
u        : -0.099
#Exceed. : 10


In [55]:
conformal_predictions_AlphaNet(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base CatBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.7901 | Size: 0.9244 | α̃: 0.9933 | ⏱ 0.01s/epoch | ETA: 1.9min


  λ=20.0 | Run 1/4 | Epoch 100/1000 | Loss: 8.9637 | Size: 4.2777 | α̃: 0.2343 | ⏱ 0.01s/epoch | ETA: 38s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 8.9394 | Size: 4.1824 | α̃: 0.2379 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 8.9394
  [Step 4/9] Training done in 38s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (CatBoost)
  Learned α̃     : 0.2325
  Half-width (r) : 2.1425
  Coverage       : 100.000%
  Avg Width      : 4.2849
  RMSE           : 0.5000
  Total time     : 38s
    ├─ Precompute: 0s
    └─ Training  : 38s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 38s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.284919,0.499972,0.1,0.767474,0.0,0.232526,20.0,38.37


# **Conformal Predictions with HistGradientBoosting**

In [56]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

In [57]:
conformal_predictions_NEXCP(
    model_class=HistGradientBoostingRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full HGBR: 5/24 test points, 0.2 min


  NexCP-Full HGBR: 10/24 test points, 0.3 min


  NexCP-Full HGBR: 15/24 test points, 0.5 min


  NexCP-Full HGBR: 20/24 test points, 0.6 min


  NexCP-Full HGBR: 24/24 test points, 0.7 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 91.667% | Width: 2.677
NexCP Full - Coverage: 100.000% | Width: 2.773
NexCP Jackknife+ aB - Coverage: 95.833% | Width: 2.447
NexCP CV+ - Coverage: 100.000% | Width: 2.716


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.916667,2.677268,0.692004,2.677268,
1,NexCP Full,1.000000,2.772885,0.692004,2.772885,1 of 24 intervals reach the edge of the y-grid...
2,NexCP Jackknife+ aB,0.958333,2.447031,0.562621,2.447031,
3,NexCP CV+,1.000000,2.716386,0.514271,2.716386,


In [58]:
conformal_predictions_online_super(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.916667  2.602780  0.692004  2.602780     
1            Online CV+  0.958333  2.531096  0.692004  2.531096     
2  Online Jackknife+ aB  0.958333  2.340451  0.565841  2.340451     
3                 SAOCP  0.875000  2.324280  0.692004  2.614815     
4                SF-OGD  0.875000  2.274025  0.692004  2.558279     
5                  FACI  0.916667  2.231013  0.692004  2.231013     
6                   ACI  0.916667  2.232091  0.692004  2.232091     
7            MFCS Split  0.916667  2.602780  0.692004  2.602780     
8             MFCS Full  0.958333  2.476095  0.470077  2.476095     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.916667,2.602780,0.692004,2.602780,
1,Online CV+,0.958333,2.531096,0.692004,2.531096,
2,Online Jackknife+ aB,0.958333,2.340451,0.565841,2.340451,
3,SAOCP,0.875000,2.324280,0.692004,2.614815,
4,SF-OGD,0.875000,2.274025,0.692004,2.558279,
5,FACI,0.916667,2.231013,0.692004,2.231013,
6,ACI,0.916667,2.232091,0.692004,2.232091,
7,MFCS Split,0.916667,2.602780,0.692004,2.602780,
8,MFCS Full,0.958333,2.476095,0.470077,2.476095,


In [59]:
conformal_predictions_COP(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (HGBR) =====
Coverage : 91.667%
Avg Width: 2.069
RMSE     : 0.692
Final q  : 0.937


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.916667,2.069057,0.692004,0.1,0.05,0.5


In [60]:
conformal_predictions_EXTREME(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
             0.6    -0.104732             10 1.131685 -1.040848     0.725687

===== EXTREME CONFORMAL PREDICTION (HGBR) =====
Coverage : 79.167%
Width    : 0.726
RMSE     : 0.564
q_extreme: 0.726
u        : -0.105
#Exceed. : 10


In [61]:
conformal_predictions_AlphaNet(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base HGBR model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/1000 | Loss: 20.8120 | Size: 0.9463 | α̃: 0.9933 | ⏱ 0.01s/epoch | ETA: 1.9min


  λ=20.0 | Run 1/4 | Epoch 100/1000 | Loss: 9.0763 | Size: 4.3378 | α̃: 0.2369 | ⏱ 0.01s/epoch | ETA: 38s


... (43 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 900/1000 | Loss: 9.0684 | Size: 4.3852 | α̃: 0.2342 | ⏱ 0.01s/epoch | ETA: 1s


  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 9.0465 | Size: 4.2727 | α̃: 0.2387 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 9.0465
  [Step 4/9] Training done in 38s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (HGBR)
  Learned α̃     : 0.2375
  Half-width (r) : 2.1443
  Coverage       : 100.000%
  Avg Width      : 4.2887
  RMSE           : 0.6020
  Total time     : 38s
    ├─ Precompute: 0s
    └─ Training  : 38s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...
  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 38s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.288676,0.602038,0.1,0.762496,0.0,0.237504,20.0,38.3


# **Conformal Predictions with PGBM**

In [62]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [63]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual str to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 1.2878
Estimator 1/300, Train metric: 1.1895
Estimator 2/300, Train metric: 1.1113
Estimator 3/300, Train metric: 1.0241
Estimator 4/300, Train metric: 0.9480
Estimator 5/300, Train metric: 0.8808
Estimator 6/300, Train metric: 0.8264
Estimator 7/300, Train metric: 0.7738
Estimator 8/300, Train metric: 0.7184
Estimator 9/300, Train metric: 0.6700
Estimator 10/300, Train metric: 0.6375
Estimator 11/300, Train metric: 0.5968
Estimator 12/300, Train metric: 0.5604
Estimator 13/300, Train metric: 0.5311
Estimator 14/300, Train metric: 0.5040
Estimator 15/300, Train metric: 0.4810
Estimator 16/300, Train metric: 0.4570


Estimator 17/300, Train metric: 0.4311
Estimator 18/300, Train metric: 0.4174
Estimator 19/300, Train metric: 0.4061
Estimator 20/300, Train metric: 0.3897
Estimator 21/300, Train metric: 0.3722
Estimator 22/300, Train metric: 0.3613
Estimator 23/300, Train metric: 0.3421
Estimator 24/300, Train metric: 0.3281
Estimator 25/300, Train metric: 0.3154
Estimator 26/300, Train metric: 0.3063
Estimator 27/300, Train metric: 0.3027
Estimator 28/300, Train metric: 0.2931
Estimator 29/300, Train metric: 0.2844
Estimator 30/300, Train metric: 0.2784
Estimator 31/300, Train metric: 0.2742
Estimator 32/300, Train metric: 0.2692
Estimator 33/300, Train metric: 0.2635
Estimator 34/300, Train metric: 0.2571
Estimator 35/300, Train metric: 0.2541
Estimator 36/300, Train metric: 0.2461
Estimator 37/300, Train metric: 0.2396
Estimator 38/300, Train metric: 0.2347
Estimator 39/300, Train metric: 0.2311
Estimator 40/300, Train metric: 0.2257
Estimator 41/300, Train metric: 0.2239
Estimator 42/300, Train m

Estimator 71/300, Train metric: 0.1922
Estimator 72/300, Train metric: 0.1922
Estimator 73/300, Train metric: 0.1922
Estimator 74/300, Train metric: 0.1922
Estimator 75/300, Train metric: 0.1922
Estimator 76/300, Train metric: 0.1922
Estimator 77/300, Train metric: 0.1922
Estimator 78/300, Train metric: 0.1922
Estimator 79/300, Train metric: 0.1922
Estimator 80/300, Train metric: 0.1922
Estimator 81/300, Train metric: 0.1922
Estimator 82/300, Train metric: 0.1870
Estimator 83/300, Train metric: 0.1870
Estimator 84/300, Train metric: 0.1870
Estimator 85/300, Train metric: 0.1870
Estimator 86/300, Train metric: 0.1870
Estimator 87/300, Train metric: 0.1870
Estimator 88/300, Train metric: 0.1870
Estimator 89/300, Train metric: 0.1870
Estimator 90/300, Train metric: 0.1870
Estimator 91/300, Train metric: 0.1870
Estimator 92/300, Train metric: 0.1870
Estimator 93/300, Train metric: 0.1870
Estimator 94/300, Train metric: 0.1870
Estimator 95/300, Train metric: 0.1870
Estimator 96/300, Train m

Estimator 264/300, Train metric: 0.1804
Estimator 265/300, Train metric: 0.1804
Estimator 266/300, Train metric: 0.1804
Estimator 267/300, Train metric: 0.1804
Estimator 268/300, Train metric: 0.1804
Estimator 269/300, Train metric: 0.1804
Estimator 270/300, Train metric: 0.1804
Estimator 271/300, Train metric: 0.1801
Estimator 272/300, Train metric: 0.1801
Estimator 273/300, Train metric: 0.1801
Estimator 274/300, Train metric: 0.1801
Estimator 275/300, Train metric: 0.1801
Estimator 276/300, Train metric: 0.1801
Estimator 277/300, Train metric: 0.1801
Estimator 278/300, Train metric: 0.1801
Estimator 279/300, Train metric: 0.1801
Estimator 280/300, Train metric: 0.1801
Estimator 281/300, Train metric: 0.1801
Estimator 282/300, Train metric: 0.1801
Estimator 283/300, Train metric: 0.1801
Estimator 284/300, Train metric: 0.1801
Estimator 285/300, Train metric: 0.1801
Estimator 286/300, Train metric: 0.1801
Estimator 287/300, Train metric: 0.1802
Estimator 288/300, Train metric: 0.1801


In [64]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test >= int_pred[strat_name][:, 0]) & (y_test <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=10):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=10, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=10, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax





In [65]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path


import os
def to_numpy_1d(y):
    return y.to_numpy().ravel() if hasattr(y, "to_numpy") else np.asarray(y).ravel()


# FIX N2: the interpolating version below is NOT the NexCP weighted quantile (no +inf mass for the
# test point, linear interpolation between residuals). Renamed so it no longer overrides the
# corrected `weighted_quantile` in the helper cell; kept only for reference.
def _old_weighted_quantile_interp(values, weights, q):
    sorter = np.argsort(values)
    values, weights = values[sorter], weights[sorter]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(q, cdf, values)



# ============================================================
# NexCP for PGBM (Excel-safe, NexCP-style, NO RETURNS)
# ============================================================

def conformal_predictions_NEXCP_PGBM(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
    seed=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(seed)

    # ========================================================
    # data conversion (PGBM-specific)
    # ========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    STRATEGIES = {
        "NexCP-Split": "NexCP Split",
        "NexCP-Full": "NexCP Full",
        "NexCP-J+aB": "NexCP Jackknife+ aB",
        "NexCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, seed)

    # NexCP - Split, eq. (11)
    q, notes["NexCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NexCP-Split"] = yhat_test
    int_pred["NexCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NexCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NexCP-Full"] = yhat_test
    int_pred["NexCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, seed)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NexCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-J+aB"] = bag_mean
    int_pred["NexCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NexCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NexCP-CV+"] = np.c_[lo, hi]

    # ========================================================
    # Metrics + Summary
    # ========================================================
    summary_rows = []
    values_dict = {}

    print("\n===== NEXCP (PGBM) RESULTS =====")

    for s in STRATEGIES:
        ymin, ymax = int_pred[s].T
        yhat = y_pred[s]

        coverage = np.mean((yte >= ymin) & (yte <= ymax))
        width = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(yte, yhat))

        print(f"{STRATEGIES[s]} - Coverage: {coverage*100:.3f}% | Width: {width:.3f}")

        summary_rows.append({
            "Strategy": STRATEGIES[s],
            "Coverage": coverage,
            "Width": width,
            "RMSE": rmse,
        })

        values_dict[f"{s}_true"] = yte
        values_dict[f"{s}_pred"] = yhat
        values_dict[f"{s}_lower"] = ymin
        values_dict[f"{s}_upper"] = ymax

    summary_df = pd.DataFrame(summary_rows)
    summary_df["Note"] = [notes.get(s, "") for s in STRATEGIES]  # FIX N10: caps / limits per method
    values_df = pd.DataFrame(values_dict)

    # ========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # ========================================================
    idx = np.arange(len(yte))
    plot_buffers = {}

    for s in STRATEGIES:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
        ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1],
                        color="gray", alpha=0.5, label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", lw=1, label="Predicted mean")

        ax.set_title(f"{STRATEGIES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel("str")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # ========================================================
    # Excel Export (append-safe)
    # ========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary sheet ----
        sheet_name = "NEXCP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot sheets ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"NEXCP_PGBM_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values sheet ----
        sheet_name = "NEXCP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_NEXCP_PGBM(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


Training on CPU
Estimator 0/300, Train metric: 1.2878
Estimator 1/300, Train metric: 1.1895
Estimator 2/300, Train metric: 1.1113
Estimator 3/300, Train metric: 1.0241
Estimator 4/300, Train metric: 0.9480
Estimator 5/300, Train metric: 0.8808
Estimator 6/300, Train metric: 0.8264
Estimator 7/300, Train metric: 0.7738
Estimator 8/300, Train metric: 0.7184
Estimator 9/300, Train metric: 0.6700
Estimator 10/300, Train metric: 0.6375
Estimator 11/300, Train metric: 0.5968
Estimator 12/300, Train metric: 0.5604
Estimator 13/300, Train metric: 0.5311
Estimator 14/300, Train metric: 0.5040
Estimator 15/300, Train metric: 0.4810
Estimator 16/300, Train metric: 0.4570
Estimator 17/300, Train metric: 0.4311
Estimator 18/300, Train metric: 0.4174
Estimator 19/300, Train metric: 0.4061
Estimator 20/300, Train metric: 0.3897
Estimator 21/300, Train metric: 0.3722
Estimator 22/300, Train metric: 0.3613


Estimator 23/300, Train metric: 0.3421
Estimator 24/300, Train metric: 0.3281
Estimator 25/300, Train metric: 0.3154
Estimator 26/300, Train metric: 0.3063
Estimator 27/300, Train metric: 0.3027
Estimator 28/300, Train metric: 0.2931
Estimator 29/300, Train metric: 0.2844
Estimator 30/300, Train metric: 0.2784
Estimator 31/300, Train metric: 0.2742
Estimator 32/300, Train metric: 0.2692
Estimator 33/300, Train metric: 0.2635
Estimator 34/300, Train metric: 0.2571
Estimator 35/300, Train metric: 0.2541
Estimator 36/300, Train metric: 0.2461
Estimator 37/300, Train metric: 0.2396
Estimator 38/300, Train metric: 0.2347
Estimator 39/300, Train metric: 0.2311
Estimator 40/300, Train metric: 0.2257
Estimator 41/300, Train metric: 0.2239
Estimator 42/300, Train metric: 0.2179
Estimator 43/300, Train metric: 0.2173
Estimator 44/300, Train metric: 0.2123
Estimator 45/300, Train metric: 0.2115
Estimator 46/300, Train metric: 0.2070
Estimator 47/300, Train metric: 0.2071
Estimator 48/300, Train m

Estimator 118/300, Train metric: 0.1857
Estimator 119/300, Train metric: 0.1857
Estimator 120/300, Train metric: 0.1857
Estimator 121/300, Train metric: 0.1857
Estimator 122/300, Train metric: 0.1857
Estimator 123/300, Train metric: 0.1857
Estimator 124/300, Train metric: 0.1857
Estimator 125/300, Train metric: 0.1857
Estimator 126/300, Train metric: 0.1857
Estimator 127/300, Train metric: 0.1854
Estimator 128/300, Train metric: 0.1854
Estimator 129/300, Train metric: 0.1851
Estimator 130/300, Train metric: 0.1851
Estimator 131/300, Train metric: 0.1851
Estimator 132/300, Train metric: 0.1851
Estimator 133/300, Train metric: 0.1851
Estimator 134/300, Train metric: 0.1851
Estimator 135/300, Train metric: 0.1851
Estimator 136/300, Train metric: 0.1851
Estimator 137/300, Train metric: 0.1851
Estimator 138/300, Train metric: 0.1851
Estimator 139/300, Train metric: 0.1851
Estimator 140/300, Train metric: 0.1851
Estimator 141/300, Train metric: 0.1851
Estimator 142/300, Train metric: 0.1851


Training on CPU
Estimator 0/300, Train metric: 1.2680
Estimator 1/300, Train metric: 1.1846
Estimator 2/300, Train metric: 1.0995
Estimator 3/300, Train metric: 1.0174
Estimator 4/300, Train metric: 0.9507
Estimator 5/300, Train metric: 0.8864
Estimator 6/300, Train metric: 0.8269
Estimator 7/300, Train metric: 0.7793
Estimator 8/300, Train metric: 0.7360
Estimator 9/300, Train metric: 0.6870
Estimator 10/300, Train metric: 0.6470
Estimator 11/300, Train metric: 0.6201
Estimator 12/300, Train metric: 0.5813
Estimator 13/300, Train metric: 0.5483
Estimator 14/300, Train metric: 0.5165
Estimator 15/300, Train metric: 0.4917
Estimator 16/300, Train metric: 0.4739
Estimator 17/300, Train metric: 0.4525
Estimator 18/300, Train metric: 0.4341
Estimator 19/300, Train metric: 0.4189
Estimator 20/300, Train metric: 0.4070
Estimator 21/300, Train metric: 0.3917
Estimator 22/300, Train metric: 0.3768
Estimator 23/300, Train metric: 0.3632
Estimator 24/300, Train metric: 0.3531
Estimator 25/300, T

Estimator 30/300, Train metric: 0.2854
Estimator 31/300, Train metric: 0.2760
Estimator 32/300, Train metric: 0.2719
Estimator 33/300, Train metric: 0.2707
Estimator 34/300, Train metric: 0.2629
Estimator 35/300, Train metric: 0.2604
Estimator 36/300, Train metric: 0.2585
Estimator 37/300, Train metric: 0.2547
Estimator 38/300, Train metric: 0.2528
Estimator 39/300, Train metric: 0.2467
Estimator 40/300, Train metric: 0.2432
Estimator 41/300, Train metric: 0.2408
Estimator 42/300, Train metric: 0.2387
Estimator 43/300, Train metric: 0.2379
Estimator 44/300, Train metric: 0.2336
Estimator 45/300, Train metric: 0.2329
Estimator 46/300, Train metric: 0.2329
Estimator 47/300, Train metric: 0.2322
Estimator 48/300, Train metric: 0.2315
Estimator 49/300, Train metric: 0.2315
Estimator 50/300, Train metric: 0.2308
Estimator 51/300, Train metric: 0.2301
Estimator 52/300, Train metric: 0.2301
Estimator 53/300, Train metric: 0.2301
Estimator 54/300, Train metric: 0.2293
Estimator 55/300, Train m

Estimator 174/300, Train metric: 0.2057
Estimator 175/300, Train metric: 0.2045
Estimator 176/300, Train metric: 0.2045
Estimator 177/300, Train metric: 0.2045
Estimator 178/300, Train metric: 0.2045
Estimator 179/300, Train metric: 0.2042
Estimator 180/300, Train metric: 0.2042
Estimator 181/300, Train metric: 0.2042
Estimator 182/300, Train metric: 0.2042
Estimator 183/300, Train metric: 0.2042
Estimator 184/300, Train metric: 0.2042
Estimator 185/300, Train metric: 0.2042
Estimator 186/300, Train metric: 0.2042
Estimator 187/300, Train metric: 0.2042
Estimator 188/300, Train metric: 0.2042
Estimator 189/300, Train metric: 0.2042
Estimator 190/300, Train metric: 0.2042
Estimator 191/300, Train metric: 0.2042
Estimator 192/300, Train metric: 0.2042
Estimator 193/300, Train metric: 0.2042
Estimator 194/300, Train metric: 0.2042
Estimator 195/300, Train metric: 0.2042
Estimator 196/300, Train metric: 0.2042
Estimator 197/300, Train metric: 0.2042
Estimator 198/300, Train metric: 0.2042


... (3,641 outputs left out)

Estimator 223/300, Train metric: 0.1726
Estimator 224/300, Train metric: 0.1726
Estimator 225/300, Train metric: 0.1726
Estimator 226/300, Train metric: 0.1726
Estimator 227/300, Train metric: 0.1726
Estimator 228/300, Train metric: 0.1726
Estimator 229/300, Train metric: 0.1726
Estimator 230/300, Train metric: 0.1726
Estimator 231/300, Train metric: 0.1726
Estimator 232/300, Train metric: 0.1726
Estimator 233/300, Train metric: 0.1726
Estimator 234/300, Train metric: 0.1726
Estimator 235/300, Train metric: 0.1726
Estimator 236/300, Train metric: 0.1726
Estimator 237/300, Train metric: 0.1726
Estimator 238/300, Train metric: 0.1726
Estimator 239/300, Train metric: 0.1726
Estimator 240/300, Train metric: 0.1726
Estimator 241/300, Train metric: 0.1726
Estimator 242/300, Train metric: 0.1726
Estimator 243/300, Train metric: 0.1726
Estimator 244/300, Train metric: 0.1726
Estimator 245/300, Train metric: 0.1726
Estimator 246/300, Train metric: 0.1726
Estimator 247/300, Train metric: 0.1726


Estimator 9/300, Train metric: 0.6412
Estimator 10/300, Train metric: 0.6038
Estimator 11/300, Train metric: 0.5607
Estimator 12/300, Train metric: 0.5268
Estimator 13/300, Train metric: 0.4994
Estimator 14/300, Train metric: 0.4726
Estimator 15/300, Train metric: 0.4408
Estimator 16/300, Train metric: 0.4101
Estimator 17/300, Train metric: 0.3913
Estimator 18/300, Train metric: 0.3760
Estimator 19/300, Train metric: 0.3611
Estimator 20/300, Train metric: 0.3458
Estimator 21/300, Train metric: 0.3326
Estimator 22/300, Train metric: 0.3153
Estimator 23/300, Train metric: 0.3001
Estimator 24/300, Train metric: 0.2913
Estimator 25/300, Train metric: 0.2803
Estimator 26/300, Train metric: 0.2703
Estimator 27/300, Train metric: 0.2641
Estimator 28/300, Train metric: 0.2507
Estimator 29/300, Train metric: 0.2427
Estimator 30/300, Train metric: 0.2373
Estimator 31/300, Train metric: 0.2270
Estimator 32/300, Train metric: 0.2250
Estimator 33/300, Train metric: 0.2171
Estimator 34/300, Train me

Estimator 51/300, Train metric: 0.1789
Estimator 52/300, Train metric: 0.1788
Estimator 53/300, Train metric: 0.1788
Estimator 54/300, Train metric: 0.1788
Estimator 55/300, Train metric: 0.1788
Estimator 56/300, Train metric: 0.1783
Estimator 57/300, Train metric: 0.1783
Estimator 58/300, Train metric: 0.1783
Estimator 59/300, Train metric: 0.1783
Estimator 60/300, Train metric: 0.1783
Estimator 61/300, Train metric: 0.1783
Estimator 62/300, Train metric: 0.1783
Estimator 63/300, Train metric: 0.1783
Estimator 64/300, Train metric: 0.1783
Estimator 65/300, Train metric: 0.1783
Estimator 66/300, Train metric: 0.1783
Estimator 67/300, Train metric: 0.1783
Estimator 68/300, Train metric: 0.1783
Estimator 69/300, Train metric: 0.1783
Estimator 70/300, Train metric: 0.1783
Estimator 71/300, Train metric: 0.1783
Estimator 72/300, Train metric: 0.1778
Estimator 73/300, Train metric: 0.1778
Estimator 74/300, Train metric: 0.1760
Estimator 75/300, Train metric: 0.1760
Estimator 76/300, Train m

Estimator 278/300, Train metric: 0.1741
Estimator 279/300, Train metric: 0.1740
Estimator 280/300, Train metric: 0.1740
Estimator 281/300, Train metric: 0.1740
Estimator 282/300, Train metric: 0.1740
Estimator 283/300, Train metric: 0.1740
Estimator 284/300, Train metric: 0.1740
Estimator 285/300, Train metric: 0.1740
Estimator 286/300, Train metric: 0.1740
Estimator 287/300, Train metric: 0.1740
Estimator 288/300, Train metric: 0.1740
Estimator 289/300, Train metric: 0.1740
Estimator 290/300, Train metric: 0.1740
Estimator 291/300, Train metric: 0.1740
Estimator 292/300, Train metric: 0.1740
Estimator 293/300, Train metric: 0.1740
Estimator 294/300, Train metric: 0.1740
Estimator 295/300, Train metric: 0.1741
Estimator 296/300, Train metric: 0.1741
Estimator 297/300, Train metric: 0.1741
Estimator 298/300, Train metric: 0.1741
Estimator 299/300, Train metric: 0.1740

===== NEXCP (PGBM) RESULTS =====
NexCP Split - Coverage: 95.833% | Width: 2.221
NexCP Full - Coverage: 91.667% | Width:

In [66]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_online_super_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30,
    seed=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(seed)

    STRATEGIES = {
        "Online-Split": "Online Split",
        "Online-CV+": "Online CV+",
        "Online-J+aB": "Online Jackknife+ aB",
        "SAOCP": "SAOCP",
        "SF-OGD": "SF-OGD",
        "FACI": "FACI",
        "ACI": "ACI",   # FIX N5: added (Gibbs & Candes 2021, gamma = 0.005)
        "MFCS-Split": "MFCS Split",
        "MFCS-Full": "MFCS Full",
    }

    if strategies == "all":
        strategies = list(STRATEGIES.keys())

    # --------------------------------------------------------
    # Safe NumPy conversion (PGBM-specific)
    # --------------------------------------------------------
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # FIX N1, N4-N6: corrected online methods. Every method starts from OUT-OF-FOLD training residuals;
    # test points arrive one at a time and each interval is OUTPUT before that point's error is used.
    # Was: in-sample training residuals; SF-OGD with flipped gradient sign and look-ahead; "FACI" and
    # "SAOCP" were window heuristics (FACI also looked ahead); J+aB scored models on their own samples.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    y_te = np.asarray(y_test).ravel()
    n_train, n_test = len(y_tr), len(y_te)

    base_model = model_class(**model_params)
    base_model.fit(X_tr, y_tr)
    yhat_te = np.asarray(base_model.predict(X_te)).ravel()
    res_te = np.abs(y_te - yhat_te)
    R_oof, mu_te_folds, _ = _oof_fold_predictions(model_class, model_params, X_tr, y_tr, X_te, 5, seed)

    y_pred, int_pred, notes = {}, {}, {}

    def _band(name, centre, q):
        y_pred[name] = np.asarray(centre)
        int_pred[name] = np.c_[centre - q, centre + q]

    if "Online-Split" in strategies:
        q, notes["Online-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("Online-Split", yhat_te, q)

    if "Online-CV+" in strategies:
        # CV+ (Barber et al. 2021); past test points join the calibration set once their error is known
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_te_folds[:, j], np.full(j, yhat_te[j])]
            R = np.r_[R_oof, res_te[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-CV+"] = yhat_te
        int_pred["Online-CV+"] = np.c_[lo, hi]
        notes["Online-CV+"] = _cap_count_note(n_cap, n_test)

    if "Online-J+aB" in strategies:
        # J+aB (Kim, Xu & Barber 2020): out-of-bag residuals; past test points join as above
        R_oob, mu_oob, bag_mean, has = _jab_oob(model_class, model_params, X_tr, y_tr, X_te, n_bootstrap, seed)
        res_bag = np.abs(y_te - bag_mean)
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_oob[:, j], np.full(j, bag_mean[j])]
            R = np.r_[R_oob, res_bag[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-J+aB"] = bag_mean
        int_pred["Online-J+aB"] = np.c_[lo, hi]
        notes["Online-J+aB"] = _cap_count_note(n_cap, n_test)

    if "SAOCP" in strategies:
        _band("SAOCP", yhat_te, _saocp(R_oof, res_te, alpha))

    if "SF-OGD" in strategies:
        _band("SF-OGD", yhat_te, _sf_ogd(R_oof, res_te, alpha))

    if "FACI" in strategies:
        _band("FACI", yhat_te, _faci(R_oof, res_te, alpha))

    if "ACI" in strategies:
        q_seq, n_cap = _aci(R_oof, res_te, alpha, gamma=0.005)
        _band("ACI", yhat_te, q_seq)
        notes["ACI"] = "" if n_cap == 0 else f"{n_cap} of {n_test} steps had alpha_t <= 0 (infinite ACI set, capped at the largest error)"

    if "MFCS-Split" in strategies:
        # kept as written by the user (no published source found); same computation as Online-Split
        q, notes["MFCS-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("MFCS-Split", yhat_te, q)

    if "MFCS-Full" in strategies:
        # refit on all points seen so far; margin from that refit's OUT-OF-FOLD residuals (FIX N1)
        lo, hi, preds, n_cap = [], [], [], 0
        for j in range(n_test):
            X_h = np.vstack([X_tr, X_te[:j]])
            y_h = np.r_[y_tr, y_te[:j]]
            m = model_class(**model_params)
            m.fit(X_h, y_h)
            yhat = float(np.asarray(m.predict(X_te[j:j + 1])).ravel()[0])
            R_h, _, _ = _oof_fold_predictions(model_class, model_params, X_h, y_h, X_te[j:j + 1], 5, seed)
            q = weighted_quantile(R_h, np.ones(len(R_h)), 1 - alpha)
            if not np.isfinite(q):
                q, n_cap = float(np.max(R_h)), n_cap + 1
            preds.append(yhat); lo.append(yhat - q); hi.append(yhat + q)
        y_pred["MFCS-Full"] = np.asarray(preds)
        int_pred["MFCS-Full"] = np.c_[lo, hi]
        notes["MFCS-Full"] = _cap_count_note(n_cap, n_test)

    # ========================================================
    # Metrics + Summary
    # ========================================================
    summary_rows = []
    values_dict = {}

    print("\n===== ONLINE CP (PGBM) RESULTS =====")

    for s in y_pred:
        ymin, ymax = int_pred[s].T
        yhat = y_pred[s]

        coverage = np.mean((yte >= ymin) & (yte <= ymax))
        width = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(yte, yhat))

        print(f"{STRATEGIES[s]} - Coverage: {coverage*100:.3f}% | Width: {width:.3f}")

        summary_rows.append({
            "Strategy": STRATEGIES[s],
            "Coverage": coverage,
            "Width": width,
            "RMSE": rmse,
        })

        values_dict[f"{s}_true"] = yte
        values_dict[f"{s}_pred"] = yhat
        values_dict[f"{s}_lower"] = ymin
        values_dict[f"{s}_upper"] = ymax

    summary_df = pd.DataFrame(summary_rows)
    summary_df["Note"] = [notes.get(s, "") for s in y_pred]  # FIX N10: caps / limits per method
    values_df = pd.DataFrame(values_dict)

    # ========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # ========================================================
    idx = np.arange(len(yte))
    plot_buffers = {}

    for s in y_pred:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
        ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1],
                        color="gray", alpha=0.5, label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", lw=1, label="Predicted mean")

        ax.set_title(f"{STRATEGIES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel("str")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # ========================================================
    # Excel Export (append-safe)
    # ========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "ONLINECP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plots ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"ONLINECP_PGBM_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "ONLINECP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_online_super_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    strategies="all",
)


Training on CPU
Estimator 0/300, Train metric: 1.2878
Estimator 1/300, Train metric: 1.1895
Estimator 2/300, Train metric: 1.1113
Estimator 3/300, Train metric: 1.0241
Estimator 4/300, Train metric: 0.9480
Estimator 5/300, Train metric: 0.8808
Estimator 6/300, Train metric: 0.8264
Estimator 7/300, Train metric: 0.7738
Estimator 8/300, Train metric: 0.7184
Estimator 9/300, Train metric: 0.6700
Estimator 10/300, Train metric: 0.6375
Estimator 11/300, Train metric: 0.5968
Estimator 12/300, Train metric: 0.5604
Estimator 13/300, Train metric: 0.5311
Estimator 14/300, Train metric: 0.5040
Estimator 15/300, Train metric: 0.4810
Estimator 16/300, Train metric: 0.4570
Estimator 17/300, Train metric: 0.4311
Estimator 18/300, Train metric: 0.4174
Estimator 19/300, Train metric: 0.4061
Estimator 20/300, Train metric: 0.3897
Estimator 21/300, Train metric: 0.3722
Estimator 22/300, Train metric: 0.3613
Estimator 23/300, Train metric: 0.3421
Estimator 24/300, Train metric: 0.3281
Estimator 25/300, T

Estimator 26/300, Train metric: 0.3063
Estimator 27/300, Train metric: 0.3027
Estimator 28/300, Train metric: 0.2931
Estimator 29/300, Train metric: 0.2844
Estimator 30/300, Train metric: 0.2784
Estimator 31/300, Train metric: 0.2742
Estimator 32/300, Train metric: 0.2692
Estimator 33/300, Train metric: 0.2635
Estimator 34/300, Train metric: 0.2571
Estimator 35/300, Train metric: 0.2541
Estimator 36/300, Train metric: 0.2461
Estimator 37/300, Train metric: 0.2396
Estimator 38/300, Train metric: 0.2347
Estimator 39/300, Train metric: 0.2311
Estimator 40/300, Train metric: 0.2257
Estimator 41/300, Train metric: 0.2239
Estimator 42/300, Train metric: 0.2179
Estimator 43/300, Train metric: 0.2173
Estimator 44/300, Train metric: 0.2123
Estimator 45/300, Train metric: 0.2115
Estimator 46/300, Train metric: 0.2070
Estimator 47/300, Train metric: 0.2071
Estimator 48/300, Train metric: 0.2049
Estimator 49/300, Train metric: 0.2035
Estimator 50/300, Train metric: 0.2035
Estimator 51/300, Train m

Estimator 135/300, Train metric: 0.1851
Estimator 136/300, Train metric: 0.1851
Estimator 137/300, Train metric: 0.1851
Estimator 138/300, Train metric: 0.1851
Estimator 139/300, Train metric: 0.1851
Estimator 140/300, Train metric: 0.1851
Estimator 141/300, Train metric: 0.1851
Estimator 142/300, Train metric: 0.1851
Estimator 143/300, Train metric: 0.1851
Estimator 144/300, Train metric: 0.1851
Estimator 145/300, Train metric: 0.1851
Estimator 146/300, Train metric: 0.1851
Estimator 147/300, Train metric: 0.1851
Estimator 148/300, Train metric: 0.1851
Estimator 149/300, Train metric: 0.1851
Estimator 150/300, Train metric: 0.1851
Estimator 151/300, Train metric: 0.1851
Estimator 152/300, Train metric: 0.1851
Estimator 153/300, Train metric: 0.1851
Estimator 154/300, Train metric: 0.1851
Estimator 155/300, Train metric: 0.1851
Estimator 156/300, Train metric: 0.1851
Estimator 157/300, Train metric: 0.1851
Estimator 158/300, Train metric: 0.1851
Estimator 159/300, Train metric: 0.1851


Estimator 3/300, Train metric: 1.0174
Estimator 4/300, Train metric: 0.9507
Estimator 5/300, Train metric: 0.8864
Estimator 6/300, Train metric: 0.8269
Estimator 7/300, Train metric: 0.7793
Estimator 8/300, Train metric: 0.7360
Estimator 9/300, Train metric: 0.6870
Estimator 10/300, Train metric: 0.6470
Estimator 11/300, Train metric: 0.6201
Estimator 12/300, Train metric: 0.5813
Estimator 13/300, Train metric: 0.5483
Estimator 14/300, Train metric: 0.5165
Estimator 15/300, Train metric: 0.4917
Estimator 16/300, Train metric: 0.4739
Estimator 17/300, Train metric: 0.4525
Estimator 18/300, Train metric: 0.4341
Estimator 19/300, Train metric: 0.4189
Estimator 20/300, Train metric: 0.4070
Estimator 21/300, Train metric: 0.3917
Estimator 22/300, Train metric: 0.3768
Estimator 23/300, Train metric: 0.3632
Estimator 24/300, Train metric: 0.3531
Estimator 25/300, Train metric: 0.3413
Estimator 26/300, Train metric: 0.3266
Estimator 27/300, Train metric: 0.3145
Estimator 28/300, Train metric: 

Estimator 40/300, Train metric: 0.2432
Estimator 41/300, Train metric: 0.2408
Estimator 42/300, Train metric: 0.2387
Estimator 43/300, Train metric: 0.2379
Estimator 44/300, Train metric: 0.2336
Estimator 45/300, Train metric: 0.2329
Estimator 46/300, Train metric: 0.2329
Estimator 47/300, Train metric: 0.2322
Estimator 48/300, Train metric: 0.2315
Estimator 49/300, Train metric: 0.2315
Estimator 50/300, Train metric: 0.2308
Estimator 51/300, Train metric: 0.2301
Estimator 52/300, Train metric: 0.2301
Estimator 53/300, Train metric: 0.2301
Estimator 54/300, Train metric: 0.2293
Estimator 55/300, Train metric: 0.2287
Estimator 56/300, Train metric: 0.2280
Estimator 57/300, Train metric: 0.2280
Estimator 58/300, Train metric: 0.2253
Estimator 59/300, Train metric: 0.2253
Estimator 60/300, Train metric: 0.2253
Estimator 61/300, Train metric: 0.2244
Estimator 62/300, Train metric: 0.2224
Estimator 63/300, Train metric: 0.2224
Estimator 64/300, Train metric: 0.2224
Estimator 65/300, Train m

Estimator 206/300, Train metric: 0.2042
Estimator 207/300, Train metric: 0.2042
Estimator 208/300, Train metric: 0.2042
Estimator 209/300, Train metric: 0.2042
Estimator 210/300, Train metric: 0.2042
Estimator 211/300, Train metric: 0.2039
Estimator 212/300, Train metric: 0.2039
Estimator 213/300, Train metric: 0.2039
Estimator 214/300, Train metric: 0.2039
Estimator 215/300, Train metric: 0.2039
Estimator 216/300, Train metric: 0.2039
Estimator 217/300, Train metric: 0.2039
Estimator 218/300, Train metric: 0.2039
Estimator 219/300, Train metric: 0.2039
Estimator 220/300, Train metric: 0.2039
Estimator 221/300, Train metric: 0.2039
Estimator 222/300, Train metric: 0.2039
Estimator 223/300, Train metric: 0.2039
Estimator 224/300, Train metric: 0.2035
Estimator 225/300, Train metric: 0.2035
Estimator 226/300, Train metric: 0.2035
Estimator 227/300, Train metric: 0.2035
Estimator 228/300, Train metric: 0.2035
Estimator 229/300, Train metric: 0.2035
Estimator 230/300, Train metric: 0.2035


... (481 outputs left out)

Estimator 194/300, Train metric: 0.1877
Estimator 195/300, Train metric: 0.1877
Estimator 196/300, Train metric: 0.1877
Estimator 197/300, Train metric: 0.1877
Estimator 198/300, Train metric: 0.1877
Estimator 199/300, Train metric: 0.1877
Estimator 200/300, Train metric: 0.1877
Estimator 201/300, Train metric: 0.1877
Estimator 202/300, Train metric: 0.1877
Estimator 203/300, Train metric: 0.1877
Estimator 204/300, Train metric: 0.1878
Estimator 205/300, Train metric: 0.1877
Estimator 206/300, Train metric: 0.1877
Estimator 207/300, Train metric: 0.1878
Estimator 208/300, Train metric: 0.1878
Estimator 209/300, Train metric: 0.1878
Estimator 210/300, Train metric: 0.1878
Estimator 211/300, Train metric: 0.1878
Estimator 212/300, Train metric: 0.1878
Estimator 213/300, Train metric: 0.1878
Estimator 214/300, Train metric: 0.1878
Estimator 215/300, Train metric: 0.1878
Estimator 216/300, Train metric: 0.1878
Estimator 217/300, Train metric: 0.1878
Estimator 218/300, Train metric: 0.1878


Estimator 10/300, Train metric: 0.5985
Estimator 11/300, Train metric: 0.5595
Estimator 12/300, Train metric: 0.5285
Estimator 13/300, Train metric: 0.4958
Estimator 14/300, Train metric: 0.4673
Estimator 15/300, Train metric: 0.4426
Estimator 16/300, Train metric: 0.4244
Estimator 17/300, Train metric: 0.4066
Estimator 18/300, Train metric: 0.3831
Estimator 19/300, Train metric: 0.3632
Estimator 20/300, Train metric: 0.3490
Estimator 21/300, Train metric: 0.3361
Estimator 22/300, Train metric: 0.3273
Estimator 23/300, Train metric: 0.3148
Estimator 24/300, Train metric: 0.3052
Estimator 25/300, Train metric: 0.2949
Estimator 26/300, Train metric: 0.2881
Estimator 27/300, Train metric: 0.2774
Estimator 28/300, Train metric: 0.2735
Estimator 29/300, Train metric: 0.2669
Estimator 30/300, Train metric: 0.2594
Estimator 31/300, Train metric: 0.2524
Estimator 32/300, Train metric: 0.2477
Estimator 33/300, Train metric: 0.2414
Estimator 34/300, Train metric: 0.2330
Estimator 35/300, Train m

Estimator 57/300, Train metric: 0.2055
Estimator 58/300, Train metric: 0.2040
Estimator 59/300, Train metric: 0.2040
Estimator 60/300, Train metric: 0.2040
Estimator 61/300, Train metric: 0.2025
Estimator 62/300, Train metric: 0.2025
Estimator 63/300, Train metric: 0.2006
Estimator 64/300, Train metric: 0.2006
Estimator 65/300, Train metric: 0.2006
Estimator 66/300, Train metric: 0.2003
Estimator 67/300, Train metric: 0.2003
Estimator 68/300, Train metric: 0.1999
Estimator 69/300, Train metric: 0.1999
Estimator 70/300, Train metric: 0.1999
Estimator 71/300, Train metric: 0.1995
Estimator 72/300, Train metric: 0.1995
Estimator 73/300, Train metric: 0.1995
Estimator 74/300, Train metric: 0.1995
Estimator 75/300, Train metric: 0.1995
Estimator 76/300, Train metric: 0.1992
Estimator 77/300, Train metric: 0.1992
Estimator 78/300, Train metric: 0.1992
Estimator 79/300, Train metric: 0.1982
Estimator 80/300, Train metric: 0.1982
Estimator 81/300, Train metric: 0.1982
Estimator 82/300, Train m

Estimator 264/300, Train metric: 0.1914
Estimator 265/300, Train metric: 0.1914
Estimator 266/300, Train metric: 0.1914
Estimator 267/300, Train metric: 0.1914
Estimator 268/300, Train metric: 0.1914
Estimator 269/300, Train metric: 0.1914
Estimator 270/300, Train metric: 0.1914
Estimator 271/300, Train metric: 0.1914
Estimator 272/300, Train metric: 0.1914
Estimator 273/300, Train metric: 0.1914
Estimator 274/300, Train metric: 0.1914
Estimator 275/300, Train metric: 0.1914
Estimator 276/300, Train metric: 0.1906
Estimator 277/300, Train metric: 0.1906
Estimator 278/300, Train metric: 0.1906
Estimator 279/300, Train metric: 0.1905
Estimator 280/300, Train metric: 0.1906
Estimator 281/300, Train metric: 0.1905
Estimator 282/300, Train metric: 0.1906
Estimator 283/300, Train metric: 0.1905
Estimator 284/300, Train metric: 0.1905
Estimator 285/300, Train metric: 0.1905
Estimator 286/300, Train metric: 0.1905
Estimator 287/300, Train metric: 0.1905
Estimator 288/300, Train metric: 0.1905


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_COP_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    lr=0.05,
    T_burnin=50,
    scale=0.5,
    random_state=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # Safe NumPy conversion (PGBM-specific)
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # Train base PGBM model
    # =========================================================
    model = model_class(**model_params)
    model.fit(Xtr, ytr)

    y_pred_test = model.predict(Xte).ravel()
    y_pred_train = model.predict(Xtr).ravel()

    # Absolute residuals (symmetric COP)
    scores_test = np.abs(yte - y_pred_test)
    scores_train = np.abs(ytr - y_pred_train)

    T_test = len(scores_test)

    # =========================================================
    # COP initialization
    # =========================================================
    # FIX N1: start from OUT-OF-FOLD training residuals (was: in-sample residuals, too small)
    R_oof_cop, _, _ = _oof_fold_predictions(model_class, model_params, np.asarray(X_train),
                                            np.asarray(y_train).ravel(), np.asarray(X_test)[:1], 5, 123)
    init_q = np.quantile(R_oof_cop, 1 - alpha)

    qs = np.zeros(T_test)
    qts = np.zeros(T_test)
    integrators = np.zeros(T_test)
    covereds = np.zeros(T_test)

    qs[0] = init_q
    qts[0] = init_q

    # =========================================================
    # Online COP loop
    # =========================================================
    for t in range(T_test - 1):

        covereds[t] = 1 if qs[t] >= scores_test[t] else 0

        # FIX N8: step size proportional to the score scale, as in the official COP code
        # (lr x (max - min) of the last T_burnin scores). Was: a fixed lr = 0.05 in target units.
        _hist = np.r_[R_oof_cop, scores_test[:t + 1]][-T_burnin:]
        lr_t = lr * (np.max(_hist) - np.min(_hist))

        # Standard COP gradient
        grad = alpha if covereds[t] else -(1 - alpha)

        # Distribution-informed (optimistic) gradient
        if t < T_burnin:
            grad_i = 0.0
        else:
            window_s = scores_test[t - T_burnin : t]
            target = qts[t] - lr_t * grad   # FIX N8
            grad_i = np.mean(window_s <= target) - (1 - alpha)

        # Updates
        integrator = -scale * grad_i
        qts[t + 1] = qts[t] - lr_t * grad   # FIX N8
        integrators[t + 1] = lr_t * integrator   # FIX N8
        qs[t + 1] = max(0.0, qts[t + 1] + integrators[t + 1])

    covereds[-1] = 1 if qs[-1] >= scores_test[-1] else 0

    # =========================================================
    # Prediction intervals
    # =========================================================
    ymin = y_pred_test - qs
    ymax = y_pred_test + qs

    # =========================================================
    # Metrics
    # =========================================================
    rmse = np.sqrt(mean_squared_error(yte, y_pred_test))
    coverage = np.mean(covereds)
    avg_width = np.mean(ymax - ymin)

    print(f"\n===== COP (PGBM) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Width    : {avg_width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"Final q  : {qs[-1]:.3f}")

    # =========================================================
    # Tables
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "COP",
        "Coverage": coverage,
        "Width": avg_width,
        "RMSE": rmse,
        "Alpha": alpha,
        "LR": lr,
        "Scale": scale,
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true": yte,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": ymax - ymin,
        "is_covered": covereds.astype(int),
        "score": scores_test,
        "adaptive_q": qs,
    })

    # =========================================================
    # Plot (SAVE ONLY)
    # =========================================================
    idx = np.arange(len(yte))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
    ax.fill_between(idx, ymin, ymax, color="gray", alpha=0.5, label="COP Interval")
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Predicted mean")


    ax.set_title(f"COP ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel("str")
    ax.legend(loc="upper left")

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # Excel export (append-safe)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "COP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "COP_PGBM_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "COP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df

conformal_predictions_COP_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


Training on CPU
Estimator 0/300, Train metric: 1.2878
Estimator 1/300, Train metric: 1.1895
Estimator 2/300, Train metric: 1.1113
Estimator 3/300, Train metric: 1.0241
Estimator 4/300, Train metric: 0.9480
Estimator 5/300, Train metric: 0.8808
Estimator 6/300, Train metric: 0.8264
Estimator 7/300, Train metric: 0.7738
Estimator 8/300, Train metric: 0.7184
Estimator 9/300, Train metric: 0.6700
Estimator 10/300, Train metric: 0.6375
Estimator 11/300, Train metric: 0.5968
Estimator 12/300, Train metric: 0.5604
Estimator 13/300, Train metric: 0.5311
Estimator 14/300, Train metric: 0.5040
Estimator 15/300, Train metric: 0.4810
Estimator 16/300, Train metric: 0.4570
Estimator 17/300, Train metric: 0.4311
Estimator 18/300, Train metric: 0.4174
Estimator 19/300, Train metric: 0.4061
Estimator 20/300, Train metric: 0.3897


Estimator 21/300, Train metric: 0.3722
Estimator 22/300, Train metric: 0.3613
Estimator 23/300, Train metric: 0.3421
Estimator 24/300, Train metric: 0.3281
Estimator 25/300, Train metric: 0.3154
Estimator 26/300, Train metric: 0.3063
Estimator 27/300, Train metric: 0.3027
Estimator 28/300, Train metric: 0.2931
Estimator 29/300, Train metric: 0.2844
Estimator 30/300, Train metric: 0.2784
Estimator 31/300, Train metric: 0.2742
Estimator 32/300, Train metric: 0.2692
Estimator 33/300, Train metric: 0.2635
Estimator 34/300, Train metric: 0.2571
Estimator 35/300, Train metric: 0.2541
Estimator 36/300, Train metric: 0.2461
Estimator 37/300, Train metric: 0.2396
Estimator 38/300, Train metric: 0.2347
Estimator 39/300, Train metric: 0.2311
Estimator 40/300, Train metric: 0.2257
Estimator 41/300, Train metric: 0.2239
Estimator 42/300, Train metric: 0.2179
Estimator 43/300, Train metric: 0.2173
Estimator 44/300, Train metric: 0.2123
Estimator 45/300, Train metric: 0.2115
Estimator 46/300, Train m

Estimator 103/300, Train metric: 0.1866
Estimator 104/300, Train metric: 0.1861
Estimator 105/300, Train metric: 0.1857
Estimator 106/300, Train metric: 0.1857
Estimator 107/300, Train metric: 0.1857
Estimator 108/300, Train metric: 0.1857
Estimator 109/300, Train metric: 0.1857
Estimator 110/300, Train metric: 0.1857
Estimator 111/300, Train metric: 0.1857
Estimator 112/300, Train metric: 0.1857
Estimator 113/300, Train metric: 0.1857
Estimator 114/300, Train metric: 0.1857
Estimator 115/300, Train metric: 0.1857
Estimator 116/300, Train metric: 0.1857
Estimator 117/300, Train metric: 0.1857
Estimator 118/300, Train metric: 0.1857
Estimator 119/300, Train metric: 0.1857
Estimator 120/300, Train metric: 0.1857
Estimator 121/300, Train metric: 0.1857
Estimator 122/300, Train metric: 0.1857
Estimator 123/300, Train metric: 0.1857
Estimator 124/300, Train metric: 0.1857
Estimator 125/300, Train metric: 0.1857
Estimator 126/300, Train metric: 0.1857
Estimator 127/300, Train metric: 0.1854


Training on CPU
Estimator 0/300, Train metric: 1.2680
Estimator 1/300, Train metric: 1.1846
Estimator 2/300, Train metric: 1.0995
Estimator 3/300, Train metric: 1.0174
Estimator 4/300, Train metric: 0.9507
Estimator 5/300, Train metric: 0.8864
Estimator 6/300, Train metric: 0.8269
Estimator 7/300, Train metric: 0.7793
Estimator 8/300, Train metric: 0.7360
Estimator 9/300, Train metric: 0.6870
Estimator 10/300, Train metric: 0.6470
Estimator 11/300, Train metric: 0.6201
Estimator 12/300, Train metric: 0.5813
Estimator 13/300, Train metric: 0.5483
Estimator 14/300, Train metric: 0.5165
Estimator 15/300, Train metric: 0.4917
Estimator 16/300, Train metric: 0.4739
Estimator 17/300, Train metric: 0.4525
Estimator 18/300, Train metric: 0.4341
Estimator 19/300, Train metric: 0.4189
Estimator 20/300, Train metric: 0.4070
Estimator 21/300, Train metric: 0.3917
Estimator 22/300, Train metric: 0.3768
Estimator 23/300, Train metric: 0.3632
Estimator 24/300, Train metric: 0.3531
Estimator 25/300, T

Estimator 33/300, Train metric: 0.2707
Estimator 34/300, Train metric: 0.2629
Estimator 35/300, Train metric: 0.2604
Estimator 36/300, Train metric: 0.2585
Estimator 37/300, Train metric: 0.2547
Estimator 38/300, Train metric: 0.2528
Estimator 39/300, Train metric: 0.2467
Estimator 40/300, Train metric: 0.2432
Estimator 41/300, Train metric: 0.2408
Estimator 42/300, Train metric: 0.2387
Estimator 43/300, Train metric: 0.2379
Estimator 44/300, Train metric: 0.2336
Estimator 45/300, Train metric: 0.2329
Estimator 46/300, Train metric: 0.2329
Estimator 47/300, Train metric: 0.2322
Estimator 48/300, Train metric: 0.2315
Estimator 49/300, Train metric: 0.2315
Estimator 50/300, Train metric: 0.2308
Estimator 51/300, Train metric: 0.2301
Estimator 52/300, Train metric: 0.2301
Estimator 53/300, Train metric: 0.2301
Estimator 54/300, Train metric: 0.2293
Estimator 55/300, Train metric: 0.2287
Estimator 56/300, Train metric: 0.2280
Estimator 57/300, Train metric: 0.2280
Estimator 58/300, Train m

Estimator 193/300, Train metric: 0.2042
Estimator 194/300, Train metric: 0.2042
Estimator 195/300, Train metric: 0.2042
Estimator 196/300, Train metric: 0.2042
Estimator 197/300, Train metric: 0.2042
Estimator 198/300, Train metric: 0.2042
Estimator 199/300, Train metric: 0.2042
Estimator 200/300, Train metric: 0.2042
Estimator 201/300, Train metric: 0.2042
Estimator 202/300, Train metric: 0.2042
Estimator 203/300, Train metric: 0.2042
Estimator 204/300, Train metric: 0.2042
Estimator 205/300, Train metric: 0.2042
Estimator 206/300, Train metric: 0.2042
Estimator 207/300, Train metric: 0.2042
Estimator 208/300, Train metric: 0.2042
Estimator 209/300, Train metric: 0.2042
Estimator 210/300, Train metric: 0.2042
Estimator 211/300, Train metric: 0.2039
Estimator 212/300, Train metric: 0.2039
Estimator 213/300, Train metric: 0.2039
Estimator 214/300, Train metric: 0.2039
Estimator 215/300, Train metric: 0.2039
Estimator 216/300, Train metric: 0.2039
Estimator 217/300, Train metric: 0.2039


... (7 outputs left out)

Estimator 261/300, Train metric: 0.1946
Estimator 262/300, Train metric: 0.1946
Estimator 263/300, Train metric: 0.1946
Estimator 264/300, Train metric: 0.1927
Estimator 265/300, Train metric: 0.1927
Estimator 266/300, Train metric: 0.1927
Estimator 267/300, Train metric: 0.1927
Estimator 268/300, Train metric: 0.1927
Estimator 269/300, Train metric: 0.1927
Estimator 270/300, Train metric: 0.1927
Estimator 271/300, Train metric: 0.1927
Estimator 272/300, Train metric: 0.1927
Estimator 273/300, Train metric: 0.1927
Estimator 274/300, Train metric: 0.1927
Estimator 275/300, Train metric: 0.1927
Estimator 276/300, Train metric: 0.1927
Estimator 277/300, Train metric: 0.1927
Estimator 278/300, Train metric: 0.1927
Estimator 279/300, Train metric: 0.1905
Estimator 280/300, Train metric: 0.1905
Estimator 281/300, Train metric: 0.1905
Estimator 282/300, Train metric: 0.1905
Estimator 283/300, Train metric: 0.1905
Estimator 284/300, Train metric: 0.1905
Estimator 285/300, Train metric: 0.1905


Estimator 21/300, Train metric: 0.3298
Estimator 22/300, Train metric: 0.3137
Estimator 23/300, Train metric: 0.2996
Estimator 24/300, Train metric: 0.2880
Estimator 25/300, Train metric: 0.2787
Estimator 26/300, Train metric: 0.2728
Estimator 27/300, Train metric: 0.2669
Estimator 28/300, Train metric: 0.2576
Estimator 29/300, Train metric: 0.2497
Estimator 30/300, Train metric: 0.2468
Estimator 31/300, Train metric: 0.2419
Estimator 32/300, Train metric: 0.2367
Estimator 33/300, Train metric: 0.2331
Estimator 34/300, Train metric: 0.2308
Estimator 35/300, Train metric: 0.2278
Estimator 36/300, Train metric: 0.2259
Estimator 37/300, Train metric: 0.2222
Estimator 38/300, Train metric: 0.2189
Estimator 39/300, Train metric: 0.2169
Estimator 40/300, Train metric: 0.2159
Estimator 41/300, Train metric: 0.2146
Estimator 42/300, Train metric: 0.2136
Estimator 43/300, Train metric: 0.2102
Estimator 44/300, Train metric: 0.2102
Estimator 45/300, Train metric: 0.2102
Estimator 46/300, Train m

Estimator 167/300, Train metric: 0.1878
Estimator 168/300, Train metric: 0.1878
Estimator 169/300, Train metric: 0.1878
Estimator 170/300, Train metric: 0.1878
Estimator 171/300, Train metric: 0.1878
Estimator 172/300, Train metric: 0.1878
Estimator 173/300, Train metric: 0.1878
Estimator 174/300, Train metric: 0.1878
Estimator 175/300, Train metric: 0.1878
Estimator 176/300, Train metric: 0.1878
Estimator 177/300, Train metric: 0.1878
Estimator 178/300, Train metric: 0.1878
Estimator 179/300, Train metric: 0.1878
Estimator 180/300, Train metric: 0.1878
Estimator 181/300, Train metric: 0.1878
Estimator 182/300, Train metric: 0.1878
Estimator 183/300, Train metric: 0.1878
Estimator 184/300, Train metric: 0.1878
Estimator 185/300, Train metric: 0.1878
Estimator 186/300, Train metric: 0.1878
Estimator 187/300, Train metric: 0.1879
Estimator 188/300, Train metric: 0.1878
Estimator 189/300, Train metric: 0.1878
Estimator 190/300, Train metric: 0.1878
Estimator 191/300, Train metric: 0.1878


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.958333,1.76944,0.492243,0.1,0.05,0.5


In [68]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_EXTREME_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    calib_size=0.2,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30,
    random_state=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # --------------------------------------------------------
    # Safe NumPy conversion (PGBM-specific)
    # --------------------------------------------------------
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # --------------------------------------------------------
    # Train / calibration split
    # --------------------------------------------------------
    X_fit, X_cal, y_fit, y_cal = train_test_split(
        Xtr, ytr, test_size=calib_size, random_state=random_state, shuffle=True
    )

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model = model_class(**model_params)
    model.fit(X_fit, y_fit)

    y_pred_test = model.predict(Xte)
    y_pred_cal = model.predict(X_cal)

    # --------------------------------------------------------
    # Upper-tail nonconformity scores
    # --------------------------------------------------------
    # FIX N9: Extreme CP as in the official ExtremeConformal code (conformalize_GPD_max):
    # signed scores y - y_hat on ALL calibration points, thresholds 0.80..0.99 (capped so >= 10 scores
    # exceed them), official GPD quantile formula, maximum over thresholds; ONE-SIDED upper bound.
    # Was: only positive scores kept, (1/alpha)^xi in the formula (missing the threshold share), a hard
    # error with < 30 exceedances, and a two-sided band reported for a one-sided method.
    scores = np.asarray(y_cal).ravel() - y_pred_cal
    q_extreme, gpd_table = _conformalize_gpd_max(scores, 1 - alpha, min_obs_gpd=10)
    extreme_note = ""
    if not np.isfinite(q_extreme):
        q_extreme = weighted_quantile(scores, np.ones(len(scores)), 1 - alpha)
        extreme_note = "GPD fit failed at every threshold; empirical conformal quantile used"
    best_row = gpd_table.loc[gpd_table["q_threshold"].idxmax()] if gpd_table["q_threshold"].notna().any() else None
    u = float(best_row["threshold_u"]) if best_row is not None else np.nan
    n_exc_used = int(best_row["n_exceedances"]) if best_row is not None else 0
    print(gpd_table.to_string(index=False))

    # one-sided interval (-inf, y_hat + q]
    y_pred_test = np.asarray(y_pred_test).ravel()
    ymin = np.array([None] * len(y_pred_test), dtype=object)   # no lower bound (empty cell in Excel)
    ymax = y_pred_test + q_extreme

    # --------------------------------------------------------
    # Metrics (one-sided coverage)
    # --------------------------------------------------------
    rmse = np.sqrt(mean_squared_error(yte, y_pred_test))
    coverage = np.mean(yte <= ymax)
    width = q_extreme

    print(f"\n===== EXTREME CP (PGBM) =====")
    print(f"Coverage     : {coverage*100:.3f}%")
    print(f"Width        : {width:.3f}")
    print(f"RMSE         : {rmse:.3f}")
    print(f"q_extreme    : {q_extreme:.3f}")
    print(f"threshold_u  : {u:.3f}")
    print(f"#Exceedances : {n_exc_used}")
    if extreme_note:
        print(f"Note     : {extreme_note}")

    # --------------------------------------------------------
    # Summary table
    # --------------------------------------------------------
    summary_df = pd.DataFrame([{
        "Strategy": "EXTREME CP",
        "Coverage": coverage,
        "Width": width,
        "RMSE": rmse,
        "q_extreme": q_extreme,
        "threshold_u": u,
        "n_exceedances": n_exc_used,
        "Note": extreme_note,   # FIX N10
    }])

    # --------------------------------------------------------
    # Values table
    # --------------------------------------------------------
    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true": yte,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": np.full(len(y_pred_test), q_extreme),   # FIX N9: one-sided width = q
        "is_covered": (yte <= ymax).astype(int),
        "residual": yte - y_pred_test,
    })

    # --------------------------------------------------------
    # Plot (SAVE ONLY — NO SHOW)
    # --------------------------------------------------------
    idx = np.arange(len(yte))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
    ax.plot(idx, ymax, ls="--", lw=1, label="Upper bound (one-sided)")   # FIX N9
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Prediction")

    ax.set_title(f"Extreme CP ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel("str")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # --------------------------------------------------------
    # Excel export (append-safe)
    # --------------------------------------------------------
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "EXTREMECP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "EXTREMECP_PGBM_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "EXTREMECP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # FIX N9: GPD fit at every threshold level (the reported q is the maximum)
        sheet_name = "EXTREMECP_PGBM_GPD_Thresholds"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        _tbl = gpd_table.astype(object).where(gpd_table.notna(), None)
        for r, row in enumerate(dataframe_to_rows(_tbl, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_EXTREME_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)

Training on CPU
Estimator 0/300, Train metric: 1.2693
Estimator 1/300, Train metric: 1.1836
Estimator 2/300, Train metric: 1.0946
Estimator 3/300, Train metric: 1.0189
Estimator 4/300, Train metric: 0.9491
Estimator 5/300, Train metric: 0.8906
Estimator 6/300, Train metric: 0.8303
Estimator 7/300, Train metric: 0.7726
Estimator 8/300, Train metric: 0.7335
Estimator 9/300, Train metric: 0.6933
Estimator 10/300, Train metric: 0.6536
Estimator 11/300, Train metric: 0.6200
Estimator 12/300, Train metric: 0.5949
Estimator 13/300, Train metric: 0.5652
Estimator 14/300, Train metric: 0.5308
Estimator 15/300, Train metric: 0.5061
Estimator 16/300, Train metric: 0.4808
Estimator 17/300, Train metric: 0.4566
Estimator 18/300, Train metric: 0.4366
Estimator 19/300, Train metric: 0.4249
Estimator 20/300, Train metric: 0.4128
Estimator 21/300, Train metric: 0.3977
Estimator 22/300, Train metric: 0.3874
Estimator 23/300, Train metric: 0.3694
Estimator 24/300, Train metric: 0.3570
Estimator 25/300, T

Estimator 37/300, Train metric: 0.2601
Estimator 38/300, Train metric: 0.2602
Estimator 39/300, Train metric: 0.2532
Estimator 40/300, Train metric: 0.2514
Estimator 41/300, Train metric: 0.2445
Estimator 42/300, Train metric: 0.2398
Estimator 43/300, Train metric: 0.2387
Estimator 44/300, Train metric: 0.2365
Estimator 45/300, Train metric: 0.2352
Estimator 46/300, Train metric: 0.2344
Estimator 47/300, Train metric: 0.2344
Estimator 48/300, Train metric: 0.2344
Estimator 49/300, Train metric: 0.2333
Estimator 50/300, Train metric: 0.2333
Estimator 51/300, Train metric: 0.2326
Estimator 52/300, Train metric: 0.2326
Estimator 53/300, Train metric: 0.2317
Estimator 54/300, Train metric: 0.2308
Estimator 55/300, Train metric: 0.2299
Estimator 56/300, Train metric: 0.2299
Estimator 57/300, Train metric: 0.2274
Estimator 58/300, Train metric: 0.2274
Estimator 59/300, Train metric: 0.2274
Estimator 60/300, Train metric: 0.2274
Estimator 61/300, Train metric: 0.2250
Estimator 62/300, Train m

Estimator 223/300, Train metric: 0.2051
Estimator 224/300, Train metric: 0.2051
Estimator 225/300, Train metric: 0.2051
Estimator 226/300, Train metric: 0.2051
Estimator 227/300, Train metric: 0.2051
Estimator 228/300, Train metric: 0.2051
Estimator 229/300, Train metric: 0.2051
Estimator 230/300, Train metric: 0.2052
Estimator 231/300, Train metric: 0.2051
Estimator 232/300, Train metric: 0.2051
Estimator 233/300, Train metric: 0.2051
Estimator 234/300, Train metric: 0.2051
Estimator 235/300, Train metric: 0.2051
Estimator 236/300, Train metric: 0.2051
Estimator 237/300, Train metric: 0.2051
Estimator 238/300, Train metric: 0.2051
Estimator 239/300, Train metric: 0.2051
Estimator 240/300, Train metric: 0.2051
Estimator 241/300, Train metric: 0.2051
Estimator 242/300, Train metric: 0.2051
Estimator 243/300, Train metric: 0.2051
Estimator 244/300, Train metric: 0.2051
Estimator 245/300, Train metric: 0.2051
Estimator 246/300, Train metric: 0.2049
Estimator 247/300, Train metric: 0.2049


In [69]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_AlphaNet_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # =========================================================
    # data conversion (PGBM-specific)
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # 1. Split training data → fit + calibration
    # =========================================================
    n_total = len(Xtr)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = Xtr[fit_idx],   ytr[fit_idx]
    X_calib, y_calib = Xtr[calib_idx],  ytr[calib_idx]
    n_calib = len(X_calib)

    alpha_min = 1.0 / (n_calib + 1) + 0.01
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =========================================================
    # 2. Train base PGBM model
    # =========================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**model_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    def _safe_predict(m, X):
        p = m.predict(X)
        return p.ravel() if hasattr(p, "ravel") else np.asarray(p).ravel()

    # =========================================================
    # 3. PRECOMPUTE all LOO scores
    # =========================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = tqdm(range(n_calib), desc="    LOO precompute", leave=False) if HAS_TQDM else range(n_calib)

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        scores_loo = np.abs(_safe_predict(base_model, loo_X) - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =========================================================
    # 4. Train AlphaNet
    # =========================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net  = None
    best_loss = np.inf
    history   = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done  = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =========================================================
    # 5. Compute conformal intervals on TEST set
    # =========================================================
    print(f"  [Step 5/9] Computing conformal intervals...")
    scores_calib = np.abs(_safe_predict(base_model, X_calib) - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small; fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = _safe_predict(base_model, Xte)
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =========================================================
    # 6. Metrics
    # =========================================================
    is_covered = ((yte >= ymin) & (yte <= ymax)).astype(int)
    coverage   = is_covered.mean()
    avg_width  = np.mean(ymax - ymin)
    rmse       = np.sqrt(mean_squared_error(yte, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =========================================================
    # 7. Summary & Values
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true":       yte,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =========================================================
    # 8. Plotting (SAVE ONLY)
    # =========================================================
    print(f"  [Step 8/9] Generating plots...")
    idx_arr = np.arange(len(yte))

    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx_arr, yte, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx_arr, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx_arr, y_pred_test, color="red", lw=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal ({model_name})")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel("str")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =========================================================
    # 9. Excel Export
    # =========================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        sn = _unique_sheet(wb, "AlphaNet_PGBM_Summary")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        img_pi = Image(buf_pi)
        sn = _unique_sheet(wb, "AlphaNet_PGBM_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(img_pi, "A1")

        img_tc = Image(buf_tc)
        sn = _unique_sheet(wb, "AlphaNet_PGBM_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(img_tc, "A1")

        sn = _unique_sheet(wb, "AlphaNet_PGBM_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet PGBM complete in {_fmt_time(total_time)}")
    return summary_df

In [70]:
conformal_predictions_AlphaNet_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base PGBM model...
Training on CPU
Estimator 0/300, Train metric: 1.2599
Estimator 1/300, Train metric: 1.1647
Estimator 2/300, Train metric: 1.0810
Estimator 3/300, Train metric: 1.0022
Estimator 4/300, Train metric: 0.9319
Estimator 5/300, Train metric: 0.8634
Estimator 6/300, Train metric: 0.8122
Estimator 7/300, Train metric: 0.7615
Estimator 8/300, Train metric: 0.7152
Estimator 9/300, Train metric: 0.6629
Estimator 10/300, Train metric: 0.6237
Estimator 11/300, Train metric: 0.5875
Estimator 12/300, Train metric: 0.5660
Estimator 13/300, Train metric: 0.5328
Estimator 14/300, Train metric: 0.5054
Estimator 15/300, Train metric: 0.4818
Estimator 16/300, Train metric: 0.4520
Estimator 17/300, Train metric: 0.4320
Estimator 18/300, Train metric: 0.4121
Estimator 19/300, Train metric: 0.3914
Estimator 20/300, Train metric: 0.3763
Estimator 21/300, Train metric: 0.3628
Estimator 22/300, Train metric: 0.3456
Estimator 23/300,

Estimator 64/300, Train metric: 0.1946
Estimator 65/300, Train metric: 0.1946
Estimator 66/300, Train metric: 0.1946
Estimator 67/300, Train metric: 0.1938
Estimator 68/300, Train metric: 0.1938
Estimator 69/300, Train metric: 0.1938
Estimator 70/300, Train metric: 0.1938
Estimator 71/300, Train metric: 0.1938
Estimator 72/300, Train metric: 0.1938
Estimator 73/300, Train metric: 0.1938
Estimator 74/300, Train metric: 0.1938
Estimator 75/300, Train metric: 0.1938
Estimator 76/300, Train metric: 0.1938
Estimator 77/300, Train metric: 0.1938
Estimator 78/300, Train metric: 0.1921
Estimator 79/300, Train metric: 0.1911
Estimator 80/300, Train metric: 0.1911
Estimator 81/300, Train metric: 0.1911
Estimator 82/300, Train metric: 0.1911
Estimator 83/300, Train metric: 0.1911
Estimator 84/300, Train metric: 0.1911
Estimator 85/300, Train metric: 0.1911
Estimator 86/300, Train metric: 0.1911
Estimator 87/300, Train metric: 0.1911
Estimator 88/300, Train metric: 0.1878
Estimator 89/300, Train m

Estimator 299/300, Train metric: 0.1780
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 41 LOO score vectors...


    LOO precompute:   0%|          | 0/41 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 1000 epochs


    Run 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

... (46 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 8.8605 | Size: 4.1489 | α̃: 0.2356 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 8.8605
  [Step 4/9] Training done in 35s
  [Step 5/9] Computing conformal intervals...

  ALPHANET CONFORMAL (PGBM)
  Learned α̃     : 0.2307
  Half-width (r) : 2.1204
  Coverage       : 100.000%
  Avg Width      : 4.2407
  RMSE           : 0.4669
  Total time     : 36s
    ├─ Precompute: 0s
    └─ Training  : 35s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet PGBM complete in 36s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,4.240732,0.466932,0.1,0.769264,0.0,0.230736,20.0,35.85


# **Conformal Predictions with TabNet**

In [71]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)

{'n_d': 8, 'n_a': 8, 'n_steps': 7, 'gamma': 2.0, 'lambda_sparse': 0.001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [72]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        y_arr = np.array(y_test)
        return {strat_name: np.mean((y_arr >= int_pred[strat_name][:, 0]) & (y_arr <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        y_arr = np.array(y_test)
        return {strat_name: np.sqrt(mean_squared_error(y_arr, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=10):
        y_arr = np.array(y_test)
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_arr[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) &
                                           (y_arr[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan)
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages
class visualize:
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        y_true_np = np.array(y_true).ravel()
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')
        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)],
                 linestyle='--', color='blue', label='Ideal Prediction Line')
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=10, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')
        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=10, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()
        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]
        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y = np.asarray(y)
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        return super().fit(X, y, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        return super().predict(X).flatten()


In [73]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_NEXCP_tabnet(
    X_train,
    y_train,
    X_test,
    y_test,
    best_params,
    model_name="TabNet",
    excel_file_path=None,
    alpha=0.1,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
):
    warnings.filterwarnings("ignore")

    SEED = 123
    rng = np.random.default_rng(SEED)
    np.random.seed(SEED)

    if "verbose" in best_params:
        best_params = dict(best_params)
        best_params.pop("verbose", None)

    model_class = TabNetRegressorCP

    strategies = {
        "NEXCP-Split": "NexCP Split",
        "NEXCP-Full": "NexCP Full",
        "NEXCP-J+aB": "NexCP Jackknife+ aB",
        "NEXCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    y_train_np = np.asarray(y_train).ravel()
    y_test_np = np.asarray(y_test).ravel()

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, SEED)

    # NexCP - Split, eq. (11)
    q, notes["NEXCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NEXCP-Split"] = yhat_test
    int_pred["NEXCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NEXCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NEXCP-Full"] = yhat_test
    int_pred["NEXCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NEXCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NEXCP-J+aB"] = bag_mean
    int_pred["NEXCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NEXCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NEXCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NEXCP-CV+"] = np.c_[lo, hi]

    # =========================
    # Metrics
    # =========================
    def coverage(y, l, u):
        return np.mean((y >= l) & (y <= u))

    def width(l, u):
        return np.mean(u - l)

    def cwc(l, u, y, alpha):
        cov = coverage(y, l, u)
        w = width(l, u)
        penalty = np.exp(-50 * (cov - (1 - alpha)))
        return w * (1 + penalty if cov < (1 - alpha) else 1)

    summary = []

    print("\n===== NEXCP TABNET RESULTS =====")
    for s in strategies:
        cov = coverage(y_test_np, int_pred[s][:, 0], int_pred[s][:, 1])
        wid = width(int_pred[s][:, 0], int_pred[s][:, 1])
        rmse = np.sqrt(mean_squared_error(y_test_np, y_pred[s]))
        cwc_val = cwc(int_pred[s][:, 0], int_pred[s][:, 1], y_test_np, alpha)

        summary.append([strategies[s], cov, wid, rmse, cwc_val])
        print(f"{strategies[s]} | Coverage: {cov*100:.3f}% | Width: {wid:.3f}")

    summary_df = pd.DataFrame(
        summary,
        columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"],
    )
    summary_df["Note"] = [notes.get(s, "") for s in strategies]  # FIX N10: caps / limits per method

    # =========================
    # Plot + Excel Export
    # =========================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        sheet = "NEXCP_TabNet_Summary"
        base = sheet
        i = 1
        while sheet in wb.sheetnames:
            sheet = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        for s in strategies:
            fig, ax = plt.subplots(figsize=(10, 4))
            idx = np.arange(len(y_test_np))

            ax.scatter(idx, y_test_np, s=10, alpha=0.7, label="True")
            ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1], alpha=0.4)
            ax.plot(idx, y_pred[s], linewidth=1, label="Pred")

            ax.set_title(f"{strategies[s]} ({model_name})")
            ax.legend()

            buf = io.BytesIO()
            fig.savefig(buf, format="png")
            buf.seek(0)
            plt.close(fig)

            img = Image(buf)
            ps = f"NEXCP_TabNet_{s}"
            base = ps
            i = 1
            while ps in wb.sheetnames:
                ps = f"{base}_{i}"
                i += 1

            ws = wb.create_sheet(ps)
            ws.add_image(img, "A1")

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


conformal_predictions_NEXCP_tabnet(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    best_params=best_params,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    decay=0.99,
    n_folds=5,
    n_bootstrap=50
)

epoch 0  | loss: 59.18275|  0:00:00s
epoch 1  | loss: 50.25857|  0:00:00s
epoch 2  | loss: 40.24729|  0:00:00s
epoch 3  | loss: 34.05163|  0:00:00s
epoch 4  | loss: 28.62308|  0:00:00s
epoch 5  | loss: 22.60353|  0:00:00s
epoch 6  | loss: 18.11546|  0:00:00s
epoch 7  | loss: 13.90322|  0:00:00s


epoch 8  | loss: 10.17948|  0:00:00s
epoch 9  | loss: 8.06678 |  0:00:00s
epoch 10 | loss: 6.14096 |  0:00:00s
epoch 11 | loss: 4.88173 |  0:00:00s
epoch 12 | loss: 3.77284 |  0:00:00s
epoch 13 | loss: 3.56025 |  0:00:00s
epoch 14 | loss: 3.72256 |  0:00:00s
epoch 15 | loss: 4.63398 |  0:00:00s


epoch 16 | loss: 4.74042 |  0:00:00s
epoch 17 | loss: 4.22344 |  0:00:00s
epoch 18 | loss: 3.91042 |  0:00:00s
epoch 19 | loss: 3.31958 |  0:00:00s
epoch 20 | loss: 2.51472 |  0:00:00s
epoch 21 | loss: 2.20546 |  0:00:00s
epoch 22 | loss: 1.69334 |  0:00:00s
epoch 23 | loss: 1.41146 |  0:00:00s


epoch 24 | loss: 1.23472 |  0:00:00s
epoch 25 | loss: 1.15903 |  0:00:00s
epoch 26 | loss: 0.90836 |  0:00:00s
epoch 27 | loss: 0.87211 |  0:00:00s
epoch 28 | loss: 0.77708 |  0:00:00s
epoch 29 | loss: 0.77617 |  0:00:00s
epoch 30 | loss: 0.77812 |  0:00:00s
epoch 31 | loss: 0.96209 |  0:00:00s


epoch 32 | loss: 0.94737 |  0:00:00s
epoch 33 | loss: 0.71488 |  0:00:01s
epoch 34 | loss: 0.71744 |  0:00:01s
epoch 35 | loss: 0.62965 |  0:00:01s
epoch 36 | loss: 0.57128 |  0:00:01s
epoch 37 | loss: 0.58553 |  0:00:01s
epoch 38 | loss: 0.53563 |  0:00:01s
epoch 39 | loss: 0.46153 |  0:00:01s


epoch 40 | loss: 0.45603 |  0:00:01s
epoch 41 | loss: 0.40423 |  0:00:01s
epoch 42 | loss: 0.34192 |  0:00:01s
epoch 43 | loss: 0.31869 |  0:00:01s
epoch 44 | loss: 0.36387 |  0:00:01s
epoch 45 | loss: 0.43583 |  0:00:01s
epoch 46 | loss: 0.44793 |  0:00:01s
epoch 47 | loss: 0.44353 |  0:00:01s


... (16,049 outputs left out)

epoch 82 | loss: 0.04813 |  0:00:02s
epoch 83 | loss: 0.04935 |  0:00:02s
epoch 84 | loss: 0.04765 |  0:00:02s
epoch 85 | loss: 0.04457 |  0:00:02s
epoch 86 | loss: 0.03795 |  0:00:02s
epoch 87 | loss: 0.03337 |  0:00:02s


epoch 88 | loss: 0.03116 |  0:00:02s
epoch 89 | loss: 0.03088 |  0:00:02s
epoch 90 | loss: 0.0299  |  0:00:02s
epoch 91 | loss: 0.02734 |  0:00:02s
epoch 92 | loss: 0.02516 |  0:00:02s
epoch 93 | loss: 0.0269  |  0:00:02s
epoch 94 | loss: 0.03318 |  0:00:02s


epoch 95 | loss: 0.02886 |  0:00:02s
epoch 96 | loss: 0.02699 |  0:00:02s
epoch 97 | loss: 0.03165 |  0:00:03s
epoch 98 | loss: 0.03636 |  0:00:03s
epoch 99 | loss: 0.03486 |  0:00:03s

===== NEXCP TABNET RESULTS =====
NexCP Split | Coverage: 87.500% | Width: 3.617
NexCP Full | Coverage: 100.000% | Width: 6.373
NexCP Jackknife+ aB | Coverage: 100.000% | Width: 2.515
NexCP CV+ | Coverage: 100.000% | Width: 4.229


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.875,3.617169,1.138356,16.242330,
1,NexCP Full,1.000,6.373108,1.138356,6.373108,14 of 24 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,1.000,2.515008,0.606104,2.515008,
3,NexCP CV+,1.000,4.228884,0.708793,4.228884,


In [74]:
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

conformal_predictions_online_super(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30
)


===== ONLINE CP RESULTS =====
               Strategy  Coverage     Width      RMSE       CWC Note
0          Online Split  0.875000  3.617169  1.138356  4.069315     
1            Online CV+  1.000000  4.408047  1.138356  4.408047     
2  Online Jackknife+ aB  0.958333  2.494577  0.626743  2.494577     
3                 SAOCP  0.916667  4.575946  1.138356  4.575946     
4                SF-OGD  0.875000  6.425329  1.138356  7.228495     
5                  FACI  0.875000  3.775676  1.138356  4.247636     
6                   ACI  0.875000  3.636223  1.138356  4.090751     
7            MFCS Split  0.875000  3.617169  1.138356  4.069315     
8             MFCS Full  0.958333  3.473238  0.780600  3.473238     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.875000,3.617169,1.138356,4.069315,
1,Online CV+,1.000000,4.408047,1.138356,4.408047,
2,Online Jackknife+ aB,0.958333,2.494577,0.626743,2.494577,
3,SAOCP,0.916667,4.575946,1.138356,4.575946,
4,SF-OGD,0.875000,6.425329,1.138356,7.228495,
5,FACI,0.875000,3.775676,1.138356,4.247636,
6,ACI,0.875000,3.636223,1.138356,4.090751,
7,MFCS Split,0.875000,3.617169,1.138356,4.069315,
8,MFCS Full,0.958333,3.473238,0.780600,3.473238,


In [75]:
conformal_predictions_COP(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    lr=0.05,
    T_burnin=50,
    scale=0.5
)


===== CONFORMAL OPTIMISTIC PREDICTION (TabNet) =====
Coverage : 87.500%
Avg Width: 3.657
RMSE     : 1.138
Final q  : 1.803


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.875,3.657317,1.138356,0.1,0.05,0.5


In [76]:
conformal_predictions_EXTREME(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.4,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30
)


 threshold_level  threshold_u  n_exceedances    sigma        xi  q_threshold
         0.69697     0.991784             10 1.035428 -0.722469     1.781622

===== EXTREME CONFORMAL PREDICTION (TabNet) =====
Coverage : 87.500%
Width    : 1.782
RMSE     : 1.071
q_extreme: 1.782
u        : 0.992
#Exceed. : 10


In [77]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_AlphaNet_tabnet(
    X_train,
    y_train,
    X_test,
    y_test,
    best_params,
    model_name="TabNet",
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    """
    Learned-α Conformal Prediction via AlphaNet — TabNet-specific variant.

    Handles:
    - TabNetRegressorCP wrapper (y reshape, predict flatten)
    - Removes 'verbose' from best_params if present
    - Precomputed LOO scores for speed
    - Progress tracking with tqdm + ETA
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # =========================================================
    # TabNet-specific: clean params & set model class
    # =========================================================
    tabnet_params = dict(best_params)
    tabnet_params.pop("verbose", None)           # TabNet doesn't use verbose here
    model_class = TabNetRegressorCP

    # =========================================================
    # data conversion
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # 1. Split training data → fit + calibration
    # =========================================================
    n_total = len(Xtr)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = Xtr[fit_idx],   ytr[fit_idx]
    X_calib, y_calib = Xtr[calib_idx],  ytr[calib_idx]
    n_calib = len(X_calib)

    # alpha_min: must satisfy α̃·(n+1) − 1 > 0  →  α̃ > 1/(n+1)
    alpha_min = 1.0 / (n_calib + 1) + 0.01
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =========================================================
    # 2. Train base TabNet model
    # =========================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**tabnet_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    # =========================================================
    # 3. PRECOMPUTE all LOO scores (KEY SPEEDUP)
    # =========================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = (
        tqdm(range(n_calib), desc="    LOO precompute", leave=False)
        if HAS_TQDM else range(n_calib)
    )

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        preds_loo = base_model.predict(loo_X).ravel()
        scores_loo = np.abs(preds_loo - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # Build feature matrix (sum of LOO scores per sample)
    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =========================================================
    # 4. Train AlphaNet (multiple runs, keep best)
    # =========================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net  = None
    best_loss = np.inf
    history   = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    # USE PRECOMPUTED scores — no predict() call here
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done  = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =========================================================
    # 5. Compute conformal intervals on TEST set
    # =========================================================
    print(f"  [Step 5/9] Computing conformal intervals...")

    calib_preds = base_model.predict(X_calib).ravel()
    scores_calib = np.abs(calib_preds - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small (denom={denom:.4f}); fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = base_model.predict(Xte).ravel()
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =========================================================
    # 6. Metrics
    # =========================================================
    is_covered = ((yte >= ymin) & (yte <= ymax)).astype(int)
    coverage   = is_covered.mean()
    avg_width  = np.mean(ymax - ymin)
    rmse       = np.sqrt(mean_squared_error(yte, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =========================================================
    # 7. Summary & Values
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true":       yte,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =========================================================
    # 8. Plotting (SAVE ONLY — NO SHOW)
    # =========================================================
    print(f"  [Step 8/9] Generating plots...")
    idx_arr = np.arange(len(yte))

    # --- Prediction-interval plot ---
    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx_arr, yte, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx_arr, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx_arr, y_pred_test, color="red", lw=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal ({model_name})")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel("str")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    # --- Training-curves plot ---
    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =========================================================
    # 9. Excel Export (append-safe)
    # =========================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        # ---- Summary ----
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Summary")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- PI Plot ----
        img_pi = Image(buf_pi)
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(img_pi, "A1")

        # ---- Training Curves Plot ----
        img_tc = Image(buf_tc)
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(img_tc, "A1")

        # ---- Values ----
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet TabNet complete in {_fmt_time(total_time)}")
    return summary_df

In [78]:
conformal_predictions_AlphaNet_tabnet(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    best_params=best_params,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/pile_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=41 | alpha_min=0.0338
  [Step 2/9] Training base TabNet model...
epoch 0  | loss: 59.644  |  0:00:00s
epoch 1  | loss: 52.57661|  0:00:00s
epoch 2  | loss: 40.22372|  0:00:00s
epoch 3  | loss: 31.37384|  0:00:00s
epoch 4  | loss: 23.19343|  0:00:00s
epoch 5  | loss: 18.23813|  0:00:00s
epoch 6  | loss: 14.13017|  0:00:00s


epoch 7  | loss: 10.06245|  0:00:00s
epoch 8  | loss: 7.66002 |  0:00:00s
epoch 9  | loss: 5.00512 |  0:00:00s
epoch 10 | loss: 3.62334 |  0:00:00s
epoch 11 | loss: 3.19618 |  0:00:00s
epoch 12 | loss: 2.46682 |  0:00:00s
epoch 13 | loss: 2.4249  |  0:00:00s
epoch 14 | loss: 2.59346 |  0:00:00s


epoch 15 | loss: 2.59421 |  0:00:00s
epoch 16 | loss: 2.29587 |  0:00:00s
epoch 17 | loss: 1.78495 |  0:00:00s
epoch 18 | loss: 1.29379 |  0:00:00s
epoch 19 | loss: 1.02325 |  0:00:00s
epoch 20 | loss: 0.93709 |  0:00:00s
epoch 21 | loss: 1.05792 |  0:00:00s
epoch 22 | loss: 1.21196 |  0:00:00s


epoch 23 | loss: 1.1698  |  0:00:00s
epoch 24 | loss: 1.04251 |  0:00:00s
epoch 25 | loss: 0.81543 |  0:00:00s
epoch 26 | loss: 0.60385 |  0:00:00s
epoch 27 | loss: 0.51589 |  0:00:00s
epoch 28 | loss: 0.59337 |  0:00:00s
epoch 29 | loss: 0.98566 |  0:00:00s
epoch 30 | loss: 0.6536  |  0:00:00s


epoch 31 | loss: 0.4889  |  0:00:00s
epoch 32 | loss: 0.62281 |  0:00:00s
epoch 33 | loss: 0.47395 |  0:00:00s
epoch 34 | loss: 0.38621 |  0:00:00s
epoch 35 | loss: 0.34285 |  0:00:00s
epoch 36 | loss: 0.29455 |  0:00:01s
epoch 37 | loss: 0.31966 |  0:00:01s
epoch 38 | loss: 0.31641 |  0:00:01s


epoch 39 | loss: 0.25472 |  0:00:01s
epoch 40 | loss: 0.18243 |  0:00:01s
epoch 41 | loss: 0.16062 |  0:00:01s
epoch 42 | loss: 0.14201 |  0:00:01s
epoch 43 | loss: 0.15415 |  0:00:01s
epoch 44 | loss: 0.28249 |  0:00:01s
epoch 45 | loss: 0.22939 |  0:00:01s
epoch 46 | loss: 0.22553 |  0:00:01s


... (56 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 999/1000 | Loss: 13.3725 | Size: 6.4244 | α̃: 0.3474 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 13.3725
  [Step 4/9] Training done in 40s
  [Step 5/9] Computing conformal intervals...

  ALPHANET CONFORMAL (TabNet)
  Learned α̃     : 0.3453
  Half-width (r) : 3.2378
  Coverage       : 100.000%
  Avg Width      : 6.4757
  RMSE           : 1.2553
  Total time     : 43s
    ├─ Precompute: 0s
    └─ Training  : 40s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet TabNet complete in 43s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,1.0,6.475676,1.255329,0.1,0.654682,0.0,0.345318,20.0,43.33
